# Multi-Stage Cyber Attack Detection on MSCAD using Feature Selection, Autoencoder and Hybrid CNN-LSTM

**Suresh** — M.Tech, Computer Networks and Information Security (CNIS)
**VNR Vignana Jyothi Institute of Engineering and Technology (VNR VJIET)**

## Objective

This notebook implements an end-to-end intrusion-detection pipeline on the MSCAD
dataset. Raw packet captures are converted into labelled network flows, merged
and preprocessed, and split into a stratified training and test set. A three-part
feature-selection stage (mutual information, Pearson redundancy filtering and
permutation importance) reduces the feature set to thirty columns, an
autoencoder compresses those to fifteen latent features, and a hybrid CNN-LSTM
classifier is trained on them with class weighting applied to the training loss
only. The trained model is evaluated on an untouched test set and compared with
three baselines, each removing one component of the proposal, so that the
contribution of each component can be attributed.


## Pipeline overview

```
PCAPNG -> flow CSV -> merge -> preprocess -> split -> MI+Pearson+PI (30)
       -> AE (15) -> class weighting -> CNN-LSTM -> evaluation -> comparison
```

| Step | Purpose | Input | Output folder |
|---|---|---|---|
| 1 | PCAPNG to one flow CSV per capture | `C:\MSCAD2022\PCAPNG_ORIGINAL` | `C:\MSCAD2022\CSVs` (reports in `Notebook\outputs\step1`) |
| 2 | Data acquisition: load the per-capture CSVs | `C:\MSCAD2022\CSVs` | `C:\MSCAD2022\outputs\step2` |
| 3 | Dataset merging | the Step 2 frames | `C:\MSCAD2022\outputs\step3` |
| 4 | Preprocessing: clean, encode, scale | `outputs\step3\MSCAD_merged.csv` | `C:\MSCAD2022\outputs\step4` |
| 5 | Train-test validation split | `outputs\step4` | `C:\MSCAD2022\outputs\step5` |
| 6 | Feature selection (MI + Pearson + PI, top 30) | `outputs\step5` | `C:\MSCAD2022\outputs\step6` |
| 7 | Autoencoder feature extraction (30 to 15) | `outputs\step6` | `C:\MSCAD2022\outputs\step7` |
| 8 | Transform the training and test sets | `outputs\step6`, `outputs\step7` | `C:\MSCAD2022\outputs\step8` |
| 9 | Training-only class weighting | `outputs\step8` | `C:\MSCAD2022\outputs\step9` |
| 10 | Hybrid CNN-LSTM training | `outputs\step8`, `outputs\step9` | `C:\MSCAD2022\outputs\step10` |
| 11 | Evaluation on the untouched test set | `outputs\step8`, `outputs\step10` | `C:\MSCAD2022\outputs\step11` |
| 12 | Model comparison against three baselines | `outputs\step5`-`step10` | `C:\MSCAD2022\outputs\step12` |

Step 1 is implemented in the companion notebook
`convert_pcapng_to_csvflows.ipynb`. Steps 2 to 12 run in this notebook.

All paths are absolute and resolve under `C:\MSCAD2022`, independent of the
working directory Jupyter was started in. Source code lives separately under
`C:\Users\banot\MSCAD22-Multi-Stage-Cyber-Attack-Detection\Notebook`; no step
writes results into the code repository.


## Contents

- [Step 2 - Data acquisition](#Step-2---Data-acquisition)
- [Step 3 - Dataset merging](#Step-3---Dataset-merging)
- [Step 4 - Preprocessing](#Step-4---Preprocessing)
- [Step 5 - Train-Test Validation Split](#Step-5---Train-Test-Validation-Split)
- [Step 6 - Feature Selection (MI + Pearson + PI to top 30)](#Step-6---Feature-Selection-(MI-+-Pearson-+-PI-to-top-30))
- [Step 7 - Autoencoder feature extraction](#Step-7---Autoencoder-feature-extraction)
- [Step 8 - Transform the training and test sets](#Step-8---Transform-the-training-and-test-sets)
- [Step 9 - Class imbalance handled in the training loss](#Step-9---Class-imbalance-handled-in-the-training-loss)
- [Step 10 - The proposed hybrid CNN-LSTM](#Step-10---The-proposed-hybrid-CNN-LSTM)
- [Step 11 - Evaluation on the untouched test set](#Step-11---Evaluation-on-the-untouched-test-set)
- [Step 12 - Model comparison](#Step-12---Model-comparison)
- [Summary](#Summary)

Each step reports its own `[OK]`/`[FAIL]` verification table. A step whose
outputs already exist is loaded rather than recomputed, unless the matching
`OVERWRITE_STEPn` flag in `config.py` is set to `True`.


## Setup

In [ ]:
import sys
from pathlib import Path

# The notebooks live in PROJECT_ROOT\notebooks, so when Jupyter is started in
# that folder the project root is one level up.  Starting Jupyter in the project
# root instead also works.  Nothing is hard-coded, so this is correct whether the
# code sits in C:\MSCAD2022\Notebook or in a git clone.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import config

print("project root :", PROJECT_ROOT)
print("kernel python:", sys.version.split()[0])
print("flow CSVs in :", config.CSV_OUTPUT_DIR)
print("merged out   :", config.MERGED_DIR)
print("step 2 out   :", config.STEP2_OUTPUT_DIR)
print("kaggle ref   :", config.KAGGLE_CSV_DIR,
      "(exists)" if config.KAGGLE_CSV_DIR.is_dir() else "(not present - section 13 is skipped)")
print("random seed  :", config.RANDOM_SEED)

---

## Step 2 - Data acquisition

**Purpose** — Load every per-capture flow CSV produced by Step 1 into a separate
DataFrame, without merging, cleaning or reordering anything.

**Input** — `<capture>_flows.csv` in `C:\MSCAD2022\CSVs`
(`config.CSV_INPUT_DIR`).

**Method** — One DataFrame per capture, each tagged with `Source_Capture` (the
file stem) and `Orig_Row` (the row's position inside its own file). The header of
every file is checked against the expected 73-column schema: 6 flow identifiers,
66 CICFlowMeter features and `Label`.

**Output** — `C:\MSCAD2022\outputs\step2` (acquisition report).

**Why** — Separating acquisition from merging makes the origin of every row
recoverable. If the folder is missing or contains no matching CSV the step stops
with a message naming the folder; a single unreadable file is reported while the
remaining files still load.


### 2.1 Load the per-capture CSVs

In [ ]:
from src.mscad import data_loading

frames, acquisition_report, failures = data_loading.acquire_captures()

print()
print("captures loaded:", list(frames))
print("failures       :", failures if failures else "none")

### 2.2 What each capture contains

One row per file: name, rows, columns, size, label counts and the first/last
timestamp. Saved to `C:\MSCAD2022\outputs\step2\acquisition_report.csv`.

In [ ]:
data_loading.show_table(acquisition_report)

---

## Step 3 - Dataset merging

**Purpose** — Concatenate the Step 2 frames into a single time-ordered dataset
and verify the saved file.

**Input** — The DataFrames held in memory from Step 2.

**Method** — Concatenation in `config.CAPTURE_ORDER`, then a stable mergesort on
`Timestamp_dt`. A stable sort preserves capture order among rows sharing a
timestamp. The merged file is written and then re-read from disk and checked.

**Output** — `C:\MSCAD2022\outputs\step3\MSCAD_merged.csv` and the Step 3
report files.

**Why** — Values are never altered here: `inf`, `NaN` and duplicate rows all
survive into Step 4, so that every later transformation is an explicit, recorded
decision rather than a side effect of merging.


### 3.1 Column consistency

Every capture must carry the same columns in the same order. A mismatch is
**reported in a table**, never silently filled -- `pd.concat` would quietly
insert `NaN` that looks like missing data later instead of like the bug it is.

In [ ]:
consistency, columns_ok = data_loading.check_column_consistency(frames)

assert columns_ok, "the captures do not share one column set -- see the table above"

### 3.2 Merge

In [ ]:
merged, _ = data_loading.merge_flow_csvs(frames=frames)

### 3.3 Merge checks - PASS/FAIL with numbers

Row totals, per-capture counts, every `Label` count, the column layout, and the
`NaN` / `inf` totals, each against the per-file numbers from Step 2.

In [ ]:
infos = [{"capture": c,
          "rows": len(f),
          "label_counts": f[config.LABEL_COLUMN].value_counts().to_dict(),
          "nan_cells": int(f[[c2 for c2 in config.KAGGLE_FEATURES if c2 in f.columns]]
                           .isna().to_numpy().sum()),
          "inf_cells": int(__import__("numpy").isinf(
              f[[c2 for c2 in config.KAGGLE_FEATURES if c2 in f.columns]]
              .to_numpy(dtype="float64")).sum()),
          "timestamp_format": None,
          "first_timestamp": f["Timestamp_dt"].min(),
          "last_timestamp": f["Timestamp_dt"].max()}
         for c, f in frames.items()]

checks, all_passed = data_loading.check_merge(merged, infos)

print()
print("ALL MERGE CHECKS PASSED:", all_passed)
assert all_passed, "the merge is not faithful -- see the FAIL rows above"

### 3.4 Save the merged dataset, and prove it is on disk

Writes `C:\MSCAD2022\outputs\step3\MSCAD_merged.csv`, creating the folder if
needed, then re-reads it and compares the row and column counts with the
in-memory frame. You get `[OK]` or a clear `[FAIL]` and the absolute path.

If the file already exists and `config.OVERWRITE_MERGED` is `False`, it is
**not** rewritten -- that is said plainly and the existing file is loaded and
verified instead.

In [ ]:
save_result = data_loading.save_merged(merged, checks=checks, infos=infos)

print()
print("status :", save_result["status"])
print("path   :", save_result["target"])
print("ok     :", save_result["ok"])
assert save_result["ok"], "the merged file is not on disk, or does not match"

### 3.5 Load the merged dataset back from disk

In [ ]:
dataset = data_loading.load_merged()

print("shape from disk:", dataset.shape)
print("matches the in-memory merge:", dataset.shape == merged.shape)

### 3.6 Step 3 report files

`label_counts.csv`, `source_capture_x_label.csv`, `label_counts.png` and
`summary.txt`, all in `C:\MSCAD2022\outputs\step3`.

In [ ]:
reports = data_loading.save_step3_reports(dataset)

### 3.7 Inspect the merged dataset

Shape and memory, `head(10)` / `tail(5)` with all columns, dtypes and null
counts, label counts, the `Source_Capture` x `Label` crosstab, per-capture
timestamps, protocols, `NaN` / `inf` per column, duplicate counts, `describe()`
with an `inf_count` column, the label chart, and the Kaggle comparison.

Everything **counts and describes** - nothing is dropped, filled or changed.

In [ ]:
summary = data_loading.summarize_dataset(dataset)

### What the merged dataset hands to Step 4

1. **`inf` in the rate columns** of zero-duration flows is arithmetically
   correct but rejected by scikit-learn and Keras, so it is capped explicitly
   rather than dropped.
2. **Identical-feature rows** are genuine observations in floods, scans and
   brute-force traffic. A random split places identical rows on both sides, which
   inflates every metric; Step 11 therefore also reports a duplicate-free score.
3. **Class imbalance** is left untouched until Step 9. Accuracy is close to a
   measure of the majority class on this dataset, so macro-F1, G-mean and
   per-class recall are the metrics to read.
4. **Identifier columns** are retained for traceability and excluded from
   training, as described above.


---

## Step 4 - Preprocessing

**Purpose** — Clean, encode and scale the whole merged dataset.

**Input** — `C:\MSCAD2022\outputs\step3\MSCAD_merged.csv`.

**Method** — `inf` values capped at each column's largest finite value with a
`zero_duration` marker (`INF_STRATEGY = "cap"`); `NaN` imputed with the column
median; zero-variance columns dropped; `Label` encoded to `Label_encoded` and a
binary `Label_binary`; `Protocol` one-hot encoded; features scaled with
`SCALER = "minmax"`.

**Output** — `C:\MSCAD2022\outputs\step4` — `features_clean.csv`,
`features_scaled.csv`, `labels.csv`, `metadata.csv`, `feature_list.json`,
`label_mapping.json` and the combined `preprocessing.csv`.

**Why** — No train/test split, no class-imbalance handling, no feature selection
and no modelling happen here, so that each of those decisions is made once, in
its own step, on the correct subset of the data.

> **Identifier columns.** `Src IP`, `Dst IP`, `Src Port`, `Dst Port`, `Protocol`
> and `Timestamp` are kept as identifiers (not model features) for traceability
> and for the optional time-window input mode; they are excluded from training to
> avoid leaking attacker identity. The attacker IP addresses determined the
> labels in Step 1, so a model trained on them would score highly without
> learning anything generalisable.

Duplicate rows are counted, never dropped: row *i* of `features`, `labels` and
`metadata` always refers to the same flow.


### Step 4 settings

In [ ]:
from src.mscad import preprocessing

print("input      :", config.MERGED_CSV)
print("output     :", config.STEP4_OUTPUT_DIR)
print("INF_STRATEGY   :", config.INF_STRATEGY, "  ('cap' or 'nan_median')")
print("SCALER         :", config.SCALER, "  ('minmax' or 'standard')")
print("LABEL_MAP      :", config.LABEL_MAP or "{} (nothing renamed)")
print("BENIGN_LABELS  :", config.BENIGN_LABELS, "-> Label_binary 0")
print("OVERWRITE_STEP4:", config.OVERWRITE_STEP4)
print("metadata kept aside:", len(config.METADATA_COLUMNS), "columns")

### 4.1 Clean

Runs in this order, each reported in its own table:

| | |
|---|---|
| **a** | `inf` -> capped at the column's largest finite value, plus a `zero_duration` marker |
| **b** | `NaN` -> median of its own column (never a blanket zero) |
| **c** | constant / zero-variance columns dropped, **measured not assumed** |
| **d** | non-feature columns moved to `metadata`, keeping row order |
| **e** | duplicate feature rows **counted, not dropped** |
| **f** | exact label strings reported; `LABEL_MAP` applied only if set |

### 4.2 Encode

`Label` -> `Label_encoded` (LabelEncoder, mapping saved as JSON) and
`Label_binary` (0 for `Benign`/`Normal`, 1 otherwise - they stay separate
classes in the multi-class target). `Protocol` -> four one-hot columns, with
anything unexpected landing in `Protocol_other`.

### 4.3 Scale

`SCALER` over the numeric features only - one-hot and binary columns keep their
0/1 meaning. **Both the scaled and the unscaled cleaned features are saved**:
this scaler is fitted on the whole dataset, which is fine for inspection but
would leak test information once a split exists, so a later step can refit it
on the training split alone from `features_clean.csv`.

### Run Step 4

In [ ]:
step4 = preprocessing.run_step4()

### What came out

`features_clean.csv`, `features_scaled.csv`, `labels.csv`, `metadata.csv`,
`label_mapping.json`, `feature_list.json`, `preprocess_artifacts.joblib` and
`step4_report.csv`, all in `C:\MSCAD2022\outputs\step4`. Every CSV was
re-read from disk and its row count compared with the frame in memory.

In [ ]:
if step4.get("status") == "loaded_existing":
    print("Existing Step 4 outputs were loaded (config.OVERWRITE_STEP4 is False).")
    print("Set OVERWRITE_STEP4 = True to recompute them.")
else:
    preprocessing.show_table(step4["report"])
    print()
    print("all checks passed:", step4["ok"])

### Row alignment

Row *i* of the features, labels and metadata refers to the same flow. This alignment
is what allows any prediction to be traced back to its originating flow.


In [ ]:
clean, labels, meta = (step4["features_clean"], step4["labels"], step4["metadata"]) \
    if "features_clean" in step4 else (None, None, None)

if clean is not None:
    print("features:", clean.shape, "| labels:", labels.shape, "| metadata:", meta.shape)
    preprocessing.show_table(
        meta.head(5).join(labels.head(5)[["Label", "Label_encoded", "Label_binary"]]))

### 4.4 Final preprocessed dataset (`preprocessing.csv`)

One file laid out for **reading** and for traceability, in the original row order:

```
Src IP | Dst IP | Src Port | Dst Port | Protocol | Timestamp | <scaled features> | Label
```

- The six identifiers carry their **original** values from the Step 3 merged
  data: IP strings unchanged, ports as integers, `Timestamp` as the original
  text. None of them is scaled.
- `Protocol` is **one readable column** - `TCP`, `UDP`, `ICMP` or `OTHER` -
  derived from the four `Protocol_*` one-hot columns. Exactly one of the four is
  `1` in every row, so the mapping is lossless and reversible, and a row that
  does not have exactly one set is reported as `[FAIL]` rather than guessed at.
  The four one-hot columns themselves are **not** in this file.
- The last column is the original `Label` string. `Label_encoded` and
  `Label_binary` are **not** here; `labels.csv` still has all three.

> **This file is the readable view, not the training matrix.** Two reasons:
> the identifier columns leak the answer - the attacker IPs decided the labels
> back in Step 1, so a model trained on them would score near-perfectly while
> learning nothing - and `Protocol` is text here, which a model cannot consume.
>
> For model training use `features_scaled.csv` (or `features_clean.csv`), which
> keep the numeric `Protocol_*` one-hot columns and exclude the identifiers
> entirely, and `labels.csv` for the encoded targets:
>
> ```python
> features = pd.read_csv(config.STEP4_OUTPUT_DIR / "features_scaled.csv")
> labels = pd.read_csv(config.STEP4_OUTPUT_DIR / "labels.csv")
> X = preprocessing.get_feature_matrix(features)   # identifiers and labels excluded
> y = labels["Label_encoded"]
> ```
>
> `config.ID_COLUMNS_NOT_FOR_TRAINING` names the columns that must never be fed
> to a model, and they are not in `feature_list.json`.

Set `config.COMBINED_ID_COLUMNS = []` for the features-and-labels-only layout,
or `COMBINED_USE_SCALED = False` to carry the unscaled features instead.


In [ ]:
import pandas as pd

combined_path = config.COMBINED_CSV
if not config.usable_output_dir(config.STEP4_OUTPUT_DIR) or not combined_path.exists():
    print("preprocessing.csv not found at", combined_path)
    print("Run the Step 4 cell above first.")
    final = None
else:
    final = pd.read_csv(combined_path, low_memory=False)
    size_mb = combined_path.stat().st_size / 1024 / 1024
    preprocessing.show_table(preprocessing.kv_table([
        ("absolute path", str(combined_path)),
        ("size (MB)", round(size_mb, 2)),
        ("rows", f"{final.shape[0]:,}"),
        ("columns", final.shape[1]),
        ("memory, deep (MB)", round(final.memory_usage(deep=True).sum() / 1024 / 1024, 1)),
    ]))

### Which columns are identifiers, features and labels

In [ ]:
if final is not None:
    ids = [c for c in config.COMBINED_ID_COLUMNS if c in final.columns]
    labels_cols = [c for c in config.COMBINED_LABEL_COLUMNS if c in final.columns]
    feature_cols = [c for c in final.columns if c not in ids + labels_cols]

    preprocessing.show_table(pd.DataFrame([
        {"role": "identifier (NOT a feature)", "columns": len(ids),
         "names": ", ".join(ids)},
        {"role": "feature (model input)", "columns": len(feature_cols),
         "names": ", ".join(feature_cols[:6]) + (", ..." if len(feature_cols) > 6 else "")},
        {"role": "label (target)", "columns": len(labels_cols),
         "names": ", ".join(labels_cols)},
        {"role": "TOTAL", "columns": final.shape[1], "names": ""},
    ]), note="`Protocol` is an identifier here, as readable text. The four numeric "
             "`Protocol_*` one-hot columns a model trains on are in "
             "`features_scaled.csv`, not in this file.")

    left_out = [c for c in config.COMBINED_DROP_FEATURE_COLUMNS if c in final.columns]
    extra = [c for c in ("Label_encoded", "Label_binary") if c in final.columns]
    print("[OK]   no Protocol_* one-hot columns" if not left_out
          else "[FAIL] found %s" % left_out)
    print("[OK]   no Label_encoded / Label_binary columns" if not extra
          else "[FAIL] found %s" % extra)

### head(10) and tail(5), all columns

In [ ]:
if final is not None:
    preprocessing.show_table(final.head(10), all_columns=True,
                             note="All %d columns; scroll sideways." % final.shape[1])
    preprocessing.show_table(final.tail(5), all_columns=True)

### Protocol counts

`Protocol` is the readable form of the four one-hot columns. The counts below
must match the one-hot sums in `features_scaled.csv` exactly - that is what the
verification table checks.


In [ ]:
if final is not None:
    counts = final["Protocol"].value_counts()
    rows = []
    for name in config.PROTOCOL_TEXT_VALUES:
        rows.append({"Protocol": name, "count": int(counts.get(name, 0)),
                     "%": round(100.0 * int(counts.get(name, 0)) / len(final), 3)})
    rows.append({"Protocol": "TOTAL", "count": len(final), "%": 100.0})
    preprocessing.show_table(pd.DataFrame(rows))

    unexpected = sorted(set(final["Protocol"].dropna().astype(str))
                        - set(config.PROTOCOL_TEXT_VALUES))
    blank = int(final["Protocol"].isna().sum())
    print("[OK]   Protocol holds only %s" % "/".join(config.PROTOCOL_TEXT_VALUES)
          if not unexpected else "[FAIL] unexpected values: %s" % unexpected)
    print("[OK]   no blank Protocol cells" if blank == 0
          else "[FAIL] %d blank Protocol cells" % blank)

### Label counts in the combined file

In [ ]:
if final is not None:
    counts = final["Label"].value_counts()
    table = pd.DataFrame({"Label": counts.index.astype(str),
                          "count": counts.to_numpy(),
                          "%": (100.0 * counts / len(final)).round(3).to_numpy()})
    table = pd.concat([table, pd.DataFrame([{"Label": "TOTAL", "count": len(final),
                                             "%": 100.0}])], ignore_index=True)
    preprocessing.show_table(table)

### Verification

Re-read from disk and checked against the other Step 4 files: the column order,
the absence of the `Protocol_*` and `Label_encoded`/`Label_binary` columns, the
identifier values against `metadata.csv` row by row, the `Protocol` counts
against the one-hot sums, the labels against `labels.csv`, and the feature values
against `features_scaled.csv`.


In [ ]:
if final is not None and "step4" in dir() and "features_scaled" in step4:
    table, ok, _ = preprocessing.verify_combined(
        final, step4["features_scaled"], step4["labels"], step4["metadata"],
        expected_rows=len(step4["features_scaled"]))
    print()
    print("ALL CHECKS PASSED:", ok)
elif final is not None:
    print("Step 4 was loaded from disk, so the in-memory frames are not available.")
    print("Re-run the Step 4 cell with OVERWRITE_STEP4 = True for the full check.")

### What Step 5 picks up

The Step 4 outputs are **unsplit and unbalanced on purpose**:

1. `features_clean.csv` is the one Step 5 splits, then fits a scaler on the
   training rows only - using `features_scaled.csv` across a split would leak
   test statistics into training.
2. The class imbalance is untouched, and Step 5 leaves it untouched too. Check
   `labels.csv`: the smallest classes have very few rows, so a stratified split
   and macro-F1 / per-class recall matter far more than accuracy.
3. `metadata.csv` holds `Src IP`, `Dst IP`, the ports and the timestamps. They
   stay **out of the feature matrix** - attacker IPs decided the labels in
   Step 1, so training on them is label leakage - but the file is kept, because
   it carries each row back to its flow and because it carries each row back to its flow.


---

## Step 5 - Train-Test Validation Split

**Purpose** — Produce a single stratified training and test split, and refit the
scaler on the training rows alone.

**Input** — `C:\MSCAD2022\outputs\step4` — `features_clean.csv` (unscaled),
`labels.csv`, `metadata.csv`.

**Method** — `train_test_split` on row positions with `TEST_SIZE = 0.20`,
`RANDOM_STATE = 42` and `STRATIFY_COLUMN = "Label"`; features, labels and
metadata are then selected with the same positions. A fresh `minmax` scaler is
fitted on the training rows only and applied to both sides.

**Output** — `C:\MSCAD2022\outputs\step5` — `X_train.csv`, `X_test.csv`,
`y_train.csv`, `y_test.csv`, `meta_train.csv`, `meta_test.csv`, the row-index
files and `scaler_train.joblib`.

**Why** — Step 4's scaler was fitted on every row, so reusing it across a split
would leak test statistics into training. Stratifying on the multiclass label
rather than the binary target keeps the smallest attack classes present on both
sides.

Min-max scaling maps each feature to $[0, 1]$:

$$x' = \frac{x - \min_{\text{train}}(x)}{\max_{\text{train}}(x) - \min_{\text{train}}(x)}$$

Test values may fall slightly outside $[0, 1]$ because the scaler never saw those
rows. This is expected, is reported per column, and is not clipped.

A single stratified 80/20 split of the Step 4 outputs, and **nothing else**.

| Step 5 does | Step 5 does not |
|---|---|
| a stratified split on row **positions**, so features, labels and metadata stay aligned | handle the class imbalance - no SMOTE, no resampling, no class weights |
| refit the scaler on the **training rows only** | select features |
| report how much of the test set is feature-identical to training | train any model |

Three settings decide everything, all in `config.py`:

```python
TEST_SIZE       = 0.20      # 20% of the rows go to test
RANDOM_STATE    = 42        # fixed, so the split is reproducible
STRATIFY_COLUMN = "Label"   # the MULTICLASS label
OVERWRITE_STEP5 = False     # keep existing outputs instead of resplitting
```

**Why stratify on `Label` and not `Label_binary`.** Stratifying on the binary
target would balance attack-vs-benign while letting a 23-row class land entirely
on one side of the split - and a class with no test rows cannot be evaluated at
all. Stratifying on the multiclass label keeps every attack type on both sides.

**Why the split is on indices.** `train_test_split` is given `np.arange(n_rows)`,
not the frames. The three frames are then selected with the same positions, so
row *i* of `X_train`, `y_train` and `meta_train` is guaranteed to be the same
flow. Splitting each frame separately would be three independent shuffles.


### Step 5 settings

In [ ]:
from src.mscad import splitting

splitting.show_table(splitting.kv_table([
    ("input folder", str(config.STEP4_OUTPUT_DIR)),
    ("output folder", str(config.STEP5_OUTPUT_DIR)),
    ("TEST_SIZE", config.TEST_SIZE),
    ("RANDOM_STATE", config.RANDOM_STATE),
    ("STRATIFY_COLUMN", config.STRATIFY_COLUMN),
    ("SCALER (refitted on train)", config.SCALER),
    ("CLIP_TEST_TO_TRAIN_RANGE", config.CLIP_TEST_TO_TRAIN_RANGE),
    ("OVERWRITE_STEP5", config.OVERWRITE_STEP5),
    ("outputs already exist", splitting.step5_outputs_exist()),
]), note="With `OVERWRITE_STEP5 = False` and the outputs already present, they "
         "are loaded instead of resplit, so the split never changes under you.")

### Run Step 5

`run_step5()` does 5.0 input checks, 5.1 the split, 5.2 the refitted scaler,
5.3 the leakage count, then writes every file and re-reads it from disk.


In [ ]:
step5 = splitting.run_step5()

### 5.1 Split - class balance

The table below is the one that matters. `share drift pp` is how far a class's
share of the training rows differs from its share of the test rows, in
percentage points. A stratified split can only move **whole rows**, so the
smallest achievable drift is one test row's worth of share; the check uses
whichever is larger, that figure or `config.SHARE_TOLERANCE_PP`.


In [ ]:
splitting.show_table(step5["balance"],
                     note="Per class: counts and shares in the full dataset, train "
                          "and test. Saved as class_balance.csv, so it is shown "
                          "whether Step 5 just ran or was loaded from disk.")

### Binary target (benign vs attack)

In [ ]:
splitting.show_table(step5["binary_balance"],
                     note="`Label_binary` is 0 for Normal/Benign and 1 for every "
                          "attack. It is NOT what the split was stratified on -- it "
                          "is shown to confirm the binary target came out balanced "
                          "as well.")

### Class counts, train vs test

Log scale, because the classes differ by four orders of magnitude. Only the test
bars carry a number: the question this chart answers is whether every class still
has test rows.


In [ ]:
chart = config.STEP5_OUTPUT_DIR / config.STEP5_FILES["chart"]
if chart.exists():
    if splitting.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(chart)))
    else:
        print("chart saved to", chart)
else:
    print("No chart at", chart)
    print("Install matplotlib and re-run Step 5 with OVERWRITE_STEP5 = True.")

### 5.2 Scale - fitted on the training rows only

Step 4's scaler saw every row, test rows included, so its min/max carry
information from the test set. Step 5 therefore starts from the **unscaled**
`features_clean.csv` and fits a fresh scaler on `X_train` alone.

Scaled test values outside `[0, 1]` are **expected**: the scaler never saw those
rows, so a test flow can be larger than every training flow. They are counted and
reported, not clipped - `CLIP_TEST_TO_TRAIN_RANGE = False`, because clipping
would quietly flatten real differences between large flows.

The one-hot `Protocol_*` columns and the `zero_duration` marker are left alone;
they are already 0/1.


In [ ]:
if "scale_info" in step5:
    info = step5["scale_info"]
    splitting.show_table(splitting.kv_table([
        ("scaler", info["scaler_name"]),
        ("fitted on", "X_train only (%s rows)" % f"{len(step5['X_train']):,}"),
        ("columns scaled", "%d of %d" % (info["n_columns"], step5["X_train"].shape[1])),
        ("columns left alone", ", ".join(
            c for c in step5["X_train"].columns if c not in set(info["columns"]))),
        ("train range", "[%.6f, %.6f]" % (info["train_min"], info["train_max"])),
        ("test range", "[%.6f, %.6f]" % (info["test_min"], info["test_max"])),
        ("test cells outside [0, 1]", f"{info.get('test_cells_outside_0_1', 0):,}"),
        ("clipped", "no" if not info["clipped"] else "yes"),
        ("saved scaler", str(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["scaler"])),
    ]))
    if len(info["out_of_range"]):
        splitting.show_table(info["out_of_range"].head(20),
                             note="Per column: how many test values fall outside "
                                  "[0, 1], and how far. Expected and small.")
else:
    # the loaded path: the same numbers, read back from split_summary.csv
    splitting.show_table(step5["split_summary"],
                         note="Read from split_summary.csv -- Step 5 was loaded from "
                              "disk, so there is no in-memory scaler report. Set "
                              "`OVERWRITE_STEP5 = True` to recompute it.")

### 5.3 Leakage check - reported, nothing dropped

About half the rows in this dataset share identical feature values once the
identifiers are removed: thousands of scan, flood and brute-force flows genuinely
look alike. A stratified random split therefore puts feature-identical twins on
both sides.

> **This inflates every test score and belongs in the results.** A model can
> "recognise" a test row it has effectively already seen in training, so the
> headline accuracy is optimistic. Report this percentage next to it, and lean on
> per-class recall for the classes whose share here is **low** - those are the
> rows the model actually had to generalise to.
>
> Nothing is dropped. These are real flows, and deleting them would change the
> dataset itself; Step 11 reports a duplicate-free score instead.


In [ ]:
splitting.show_table(step5["leakage_table"],
                     note="An exact match on the full UNSCALED feature vector. Saved "
                          "as leakage.csv.")

total = step5["leakage_table"].iloc[-1]
print("%s of %s test rows (%s%%) have a feature-identical twin in training."
      % (f"{int(total['also in train']):,}", f"{int(total['test rows']):,}",
         total["% of this class's test rows"]))

### 5.4 Saved files

Everything below is in `C:\MSCAD2022\outputs\step5\`.

| file | what it is |
|---|---|
| `X_train.csv` / `X_test.csv` | the features, scaled with the **train-fitted** scaler - this is what a model trains and is evaluated on |
| `X_train_unscaled.csv` / `X_test_unscaled.csv` | the same rows straight from `features_clean.csv`, for a model that wants its own scaling |
| `y_train.csv` / `y_test.csv` | `Label`, `Label_encoded`, `Label_binary` |
| `meta_train.csv` / `meta_test.csv` | the identifiers, including `Orig_Row`, so any row goes back to its flow |
| `train_indices.csv` / `test_indices.csv` | the row positions in the Step 4 files - the split, reproducible without rerunning it |
| `train_view.csv` / `test_view.csv` | the `preprocessing.csv` layout, **for viewing only** |
| `scaler_train.joblib` | the fitted scaler, with the settings it was fitted under |
| `split_summary.csv`, `step5_report.csv` | the numbers and the actions |
| `split_class_counts.png` | the chart above |

`X_train.csv` is the training matrix: the identifiers are not in it, and the
`Protocol_*` one-hot columns are. The two `*_view.csv` files are the readable
counterpart and must never be fed to a model.


In [ ]:
import pandas as pd

folder = config.usable_output_dir(config.STEP5_OUTPUT_DIR)
if folder is None:
    print("Step 5 outputs are not readable here:",
          config.why_unusable(config.STEP5_OUTPUT_DIR))
else:
    rows = []
    for key, name in config.STEP5_FILES.items():
        path = folder / name
        rows.append({"file": name, "exists": path.exists(),
                     "size (MB)": round(path.stat().st_size / 1024 / 1024, 3)
                                  if path.exists() else 0.0,
                     "what": key})
    splitting.show_table(pd.DataFrame(rows), note="Folder: %s" % folder)

### Shapes, and the head of each side

In [ ]:
X_train = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["X_train"],
                      low_memory=False)
X_test = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["X_test"],
                     low_memory=False)
y_train = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["y_train"])
y_test = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["y_test"])

splitting.show_table(splitting.kv_table([
    ("X_train", "%s rows x %d columns" % (f"{X_train.shape[0]:,}", X_train.shape[1])),
    ("X_test", "%s rows x %d columns" % (f"{X_test.shape[0]:,}", X_test.shape[1])),
    ("y_train", "%s rows x %d columns" % (f"{y_train.shape[0]:,}", y_train.shape[1])),
    ("y_test", "%s rows x %d columns" % (f"{y_test.shape[0]:,}", y_test.shape[1])),
    ("train + test", f"{len(X_train) + len(X_test):,}"),
    ("identical columns", list(X_train.columns) == list(X_test.columns)),
]))

counts = y_test[config.LABEL_COLUMN].value_counts()
splitting.show_table(pd.DataFrame({"Label": counts.index.astype(str),
                                   "test rows": counts.to_numpy()}),
                     note="Per-class test counts -- every class must appear here.")
splitting.show_table(X_train.head(5), all_columns=True,
                     note="X_train.head(5), all %d columns." % X_train.shape[1])

### Verification

Every file above was re-read from disk and checked: the row counts add up to the
Step 4 total, no row position is in both sides, `X_train` and `X_test` have the
same columns in the same order, the labels and metadata line up with the split
positions (compared on `Orig_Row`), and neither matrix holds a NaN or an inf.


In [ ]:
if "saved" in step5 and "verification" in step5.get("saved", {}):
    splitting.show_table(step5["saved"]["verification"])
    print()
    print("ALL CHECKS PASSED:", step5["saved"]["ok"])
else:
    print("Step 5 was loaded from disk, so there is no fresh verification table.")
    print("Set OVERWRITE_STEP5 = True and re-run to split and verify again.")

### What Step 6 picks up

1. **Train on `X_train.csv` only.** `X_test.csv` is touched once, at the end, to
   report a score. Any choice made by looking at the test set - a threshold, a
   feature, a hyper-parameter - makes that score meaningless.
2. **The imbalance is still there, on purpose.** SMOTE, resampling or class
   weights belong in Step 6 and must be fitted on the **training** side of this
   split only. Oversampling before the split copies rows across it, which is the
   same leak as scaling before the split.
3. **Use `train_indices.csv` / `test_indices.csv`** to carry this exact split
   into any other experiment, instead of resplitting with the same seed and
   hoping.
4. **Report the 5.3 leakage percentage next to every score.** Macro-F1 and
   per-class recall, not accuracy: with Brute_Force at tens of thousands of rows
   and Web_Crwling at 23, accuracy is nearly a measure of the majority class.


---

## Step 6 - Feature Selection (MI + Pearson + PI to top 30)

**Purpose** — Reduce the feature set to `N_SELECTED_FEATURES = 30` columns using
three complementary measures.

**Input** — `C:\MSCAD2022\outputs\step5` — `X_train.csv`, `y_train.csv`.
`X_test.csv` is read only at the end, to take the same columns.

**Method** — Mutual information between each feature and `Label_encoded`
(`MI_N_NEIGHBORS = 3`), computed once; a Pearson redundancy filter dropping one
of every pair with $|r| \geq 0.95$ (`PEARSON_THRESHOLD`), keeping the higher-MI
feature; permutation importance of a random forest on an inner split of the
training set (`PI_SCORING = "f1_macro"`, `PI_N_REPEATS = 10`).

**Output** — `C:\MSCAD2022\outputs\step6` — `X_train_selected.csv`,
`X_test_selected.csv`, `selected_features.json`, `combined_ranking.csv` and the
supporting tables and charts.

**Why** — The three measures fail in different directions. Mutual information
examines one feature at a time, so two identical features both score highly;
Pearson removes that redundancy without using labels; permutation importance
measures what a model actually uses, but understates a feature whose correlated
partner remains in the set.

Mutual information between feature $X$ and label $Y$:

$$I(X; Y) = \sum_{x}\sum_{y} p(x, y)\, \log \frac{p(x, y)}{p(x)\,p(y)}$$

Pearson correlation between two features:

$$r_{ij} = \frac{\operatorname{cov}(X_i, X_j)}{\sigma_{X_i}\,\sigma_{X_j}}$$

Permutation importance of feature $j$, where $s$ is macro-F1 and $s_{k,j}$ is the
score after the $k$-th shuffle of that column:

$$\mathrm{PI}_j = s - \frac{1}{K}\sum_{k=1}^{K} s_{k,j}$$

Both scores are min-max normalised over the surviving features and added, so each
contributes at most 1.0:

$$\text{CombinedScore} = \mathrm{MI}' + \mathrm{PI}', \qquad
\mathrm{MI}' = \frac{\mathrm{MI} - \min}{\max - \min}$$

66+ features in, 30 out. Three measures, each answering a different question:

| | what it measures | blind to |
|---|---|---|
| **6.2 Mutual information** | how much knowing the feature tells you about the class | every other feature - two identical features both score highly |
| **6.3 Pearson \|r\|** | how much two **features** repeat each other. No labels at all | whether either is useful |
| **6.4 Permutation importance** | how far the forest's macro-F1 falls when that one column is shuffled | a feature whose partner can stand in for it |

**Why all three, and why MI and PI are added rather than one being picked.**
They fail in opposite directions. PI understates a feature whose partner is
still in the set at \|r\| around 0.80-0.95: shuffle one and the forest simply
leans on the other, so neither looks important. MI never sees other features, so
that cannot happen to it. Conversely MI rewards a feature that merely tracks the
label without adding anything the others do not already give; PI catches those.
Adding the two normalised scores means a feature has to fail *both* tests to be
dropped.

**Everything is decided on the training set.** MI, the correlation matrix, the
inner fit/validation split and the permutation importance all run on `X_train`
alone. `X_test` is touched once, in 6.6, to take the same 30 columns in the same
order. Choosing features by looking at the test set is the same leak as scaling
before the split.

> **Step 6 does nothing else.** No autoencoder, no CNN-LSTM, no modelling. No
> class-imbalance handling - no SMOTE, no resampling. The RandomForest in 6.4
> exists only to rank features; 6.4 explains why its score is not a result.


### Step 6 settings

In [ ]:
from src.mscad import feature_selection

feature_selection.show_table(feature_selection.kv_table([
    ("input folder", str(config.STEP5_OUTPUT_DIR)),
    ("output folder", str(config.STEP6_OUTPUT_DIR)),
    ("PEARSON_THRESHOLD", config.PEARSON_THRESHOLD),
    ("PEARSON_REPORT_LOWER", config.PEARSON_REPORT_LOWER),
    ("N_SELECTED_FEATURES", config.N_SELECTED_FEATURES),
    ("RF_PARAMS", str(config.RF_PARAMS)),
    ("PI_SCORING / PI_N_REPEATS", "%s / %d" % (config.PI_SCORING, config.PI_N_REPEATS)),
    ("PI_VALIDATION_SIZE", config.PI_VALIDATION_SIZE),
    ("MI_N_NEIGHBORS", config.MI_N_NEIGHBORS),
    ("CLIP_NEGATIVE_PI", config.CLIP_NEGATIVE_PI),
    ("OVERWRITE_STEP6", config.OVERWRITE_STEP6),
    ("outputs already exist", feature_selection.step6_outputs_exist()),
]), note="`PI_SCORING = \"f1_macro\"` weights every class equally, so a feature "
         "that only the 23-row classes depend on still registers. With `accuracy` "
         "the 84% Brute_Force class would drown them out entirely.")

### Run Step 6

**This cell is slow.** Permutation importance refits nothing but re-scores the
validation rows `n_repeats` times per feature - on 66 features at 10 repeats
that is 660 scorings, and it can take several minutes. The elapsed time for MI
and for PI is printed as each finishes.

With `OVERWRITE_STEP6 = False` and the outputs already present, they are loaded
from disk instead, which is why that setting is worth keeping.


In [ ]:
step6 = feature_selection.run_step6()

### 6.1 Training data overview

The input is `X_train.csv` / `X_test.csv` from Step 5, already scaled with the
train-fitted scaler. Before anything is scored, four things are confirmed: the
two frames carry identical columns in identical order, the labels line up, no
label column has crept into the features, and no identifier has either.

A column that is **constant** across the training rows is dropped here. Pearson
correlation is undefined for it (zero variance means dividing by zero), so it
would otherwise put a row and a column of `NaN` into the matrix and quietly
corrupt the redundancy filter. A constant also tells a model nothing, so nothing
is lost.


In [ ]:
if step6.get("status") != "loaded_existing":
    feature_selection.show_table(step6["classes"])
    dropped = step6["constant_dropped"]
    print("constant columns dropped:", ", ".join(dropped) if dropped else "(none)")
    print("features after the drop  :", len(step6["mi"]))
else:
    print("Step 6 was loaded from disk, so the in-memory tables are not available.")
    print("Set OVERWRITE_STEP6 = True in config.py and re-run the cell above.")

### 6.2 Mutual information

`mutual_info_classif` between every feature and `Label_encoded`, on the training
rows. MI is a *statistical dependency* measure: unlike a correlation it finds
non-linear relationships, and it makes no assumption that the relationship has a
direction.

Two details that matter:

- The one-hot and binary columns (`Protocol_*`, `zero_duration`) are declared
  **discrete**. Left as continuous, the k-nearest-neighbour estimator would
  treat a column of 0s and 1s as a dense number line and return a wrong score.
- MI is **stochastic** - it jitters continuous inputs before estimating - so it
  is computed **once** per run and the same numbers are reused for the Pearson
  tie-break in 6.3 and the ranking in 6.5. Recomputing it would give slightly
  different values and the two stages would silently disagree.


In [ ]:
if step6.get("status") != "loaded_existing":
    feature_selection.show_table(step6["mi_table_all"].head(25),
                                 note="Top 25 of %d by mutual information. Saved in "
                                      "full as mi_scores_all.csv."
                                      % len(step6["mi_table_all"]))
    print("MI computed in %.1f s" % step6["mi_seconds"])
else:
    feature_selection.show_table(step6["mi_all"].head(25),
                                 note="Read back from mi_scores_all.csv.")

### 6.3 Pearson redundancy filter

The absolute correlation between every **pair of features**. No labels are
involved: this step is only about features repeating each other.

For every pair at `|r| >= 0.95` one of the two is dropped, keeping whichever has
the higher MI from 6.2. The walk is deterministic by construction: pairs are
processed from the highest `|r|` downward, a pair is skipped once either side has
already gone, and an MI tie falls back to feature order - so the result never
depends on dictionary or set iteration order.

CICFlowMeter produces several near-copies by design - `Subflow Fwd Pkts` is
usually `Tot Fwd Pkts`, `Pkt Size Avg` tracks `Pkt Len Mean` - which is what this
filter is for. Expect the `Protocol_*` one-hot columns to show up here too: with
three protocols, two of the indicators determine the third, so they are genuinely
redundant.


In [ ]:
if step6.get("status") != "loaded_existing":
    dropped = step6["dropped_pairs"]
    if len(dropped):
        feature_selection.show_table(dropped,
            note="`MI kept` is always >= `MI dropped` - the higher-MI half of each "
                 "pair survives. Saved as pearson_dropped_pairs.csv.")
    else:
        print("No pair reached |r| >= %.2f, so nothing was dropped."
              % config.PEARSON_THRESHOLD)
    print("features: %d  ->  %d" % (len(step6["mi"]), len(step6["pearson_kept"])))

### Still correlated, but kept

These pairs sit in `[0.80, 0.95)`. They are not redundant enough to drop, but
they are exactly the pairs permutation importance will **understate**: shuffle
one and the forest leans on the other, so neither looks important. That is the
concrete reason MI is added to PI rather than replaced by it.


In [ ]:
if step6.get("status") != "loaded_existing":
    remaining = step6["remaining_pairs"]
    if len(remaining):
        feature_selection.show_table(remaining.head(25),
            note="%d pair(s) in [%.2f, %.2f). Saved as pearson_remaining_pairs.csv."
                 % (len(remaining), config.PEARSON_REPORT_LOWER,
                    config.PEARSON_THRESHOLD))
    else:
        print("No remaining pair in [%.2f, %.2f)."
              % (config.PEARSON_REPORT_LOWER, config.PEARSON_THRESHOLD))

### Correlation heatmap

One hue, light to dark: `|r|` is a magnitude, so a single blue ramp reads it
correctly. The dark diagonal is each feature against itself; any other dark
square is a redundant pair.


In [ ]:
heatmap = config.STEP6_OUTPUT_DIR / config.STEP6_FILES["heatmap"]
if heatmap.exists():
    if feature_selection.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(heatmap)))
    else:
        print("heatmap saved to", heatmap)
else:
    print("No heatmap at", heatmap)
    print("Install matplotlib and re-run Step 6 with OVERWRITE_STEP6 = True.")

### 6.4 Permutation importance

The training set is split **again**, into an inner fit part and an inner
validation part (stratified, 80/20). A RandomForest is fitted on the fit part;
then each feature's column in the validation part is shuffled and the drop in
macro-F1 is measured. A feature the forest relies on loses it a lot of score; a
feature it ignores loses it nothing.

> **The baseline macro-F1 below is a ranking tool, not a model result.** Around
> half of this dataset's rows share identical feature values once the identifiers
> are gone - thousands of scan, flood and brute-force flows genuinely look alike -
> so feature-identical rows sit in both the fit and the validation part. The
> forest is scored partly on rows it has already seen. The percentage is printed
> below; quote Step 7's numbers, never this one.

> **PI understates correlated features.** For any pair still at `|r|` 0.80-0.95
> after 6.3, shuffling one leaves the other intact and the forest simply uses it,
> so both score low. MI is immune to this, which is the whole reason 6.5 adds the
> two instead of trusting either alone.


In [ ]:
if step6.get("status") != "loaded_existing":
    info = step6["pi_info"]
    feature_selection.show_table(feature_selection.kv_table([
        ("inner fit rows", f"{info['fit_rows']:,}"),
        ("inner validation rows", f"{info['validation_rows']:,}"),
        ("forest fit time", "%.1f s" % info["fit_seconds"]),
        ("permutation time", "%.1f s" % info["pi_seconds"]),
        ("scoring", "%s, %d repeats" % (config.PI_SCORING, config.PI_N_REPEATS)),
        ("baseline macro-F1", "%.4f" % info["baseline_macro_f1"]),
        ("validation rows also in the fit part", "%s (%.2f%%)"
         % (f"{info['overlap']['also in fit']:,}", info["overlap"]["% of validation"])),
    ]))
    feature_selection.show_table(info["per_class_f1"],
                                 note="Per-class F1 on the inner validation part.")
    if info["classes_missing_from_validation"]:
        print("[WARN] no validation rows for:",
              ", ".join(info["classes_missing_from_validation"]))
    if info["thin_validation_classes"]:
        print("[WARN] fewer than 2 validation rows for:",
              info["thin_validation_classes"])
    feature_selection.show_table(step6["pi_table"].head(25),
                                 note="Top 25 by permutation importance. Saved in "
                                      "full as pi_scores.csv.")
else:
    feature_selection.show_table(step6["pi_scores"].head(25),
                                 note="Read back from pi_scores.csv.")

### How much of the inner validation part is already in the fit part

In [ ]:
if step6.get("status") != "loaded_existing":
    feature_selection.show_table(step6["overlap_table"],
        note="Exact matches on the full feature vector. The classes with a LOW "
             "percentage here are the ones the forest actually had to generalise "
             "to - weigh their F1 accordingly.")
else:
    feature_selection.show_table(step6["pi_overlap"],
                                 note="Read back from pi_duplicate_overlap.csv.")

### 6.5 Combine MI and PI

The two scores live on completely different scales - MI is in nats, PI is a
difference in F1 - so adding them raw would let whichever happens to be larger
decide everything. Both are min-max normalised across the surviving features
first:

$$\text{MI}' = \frac{\text{MI} - \min}{\max - \min}, \qquad
  \text{PI}' = \frac{\text{PI} - \min}{\max - \min}, \qquad
  \text{CombinedScore} = \text{MI}' + \text{PI}'$$

Each part contributes at most 1.0, so the combined score runs 0 to 2 and neither
measure can dominate by living on a bigger scale. Two edge cases are handled
explicitly rather than left to produce `NaN`: a **negative** PI (shuffling made
the score *better*, which is noise around zero, not evidence against the feature)
is clipped to 0 so it cannot stretch the scale, and a **flat** score where
`max == min` normalises to 0 for every feature with a warning, leaving the
ranking to the other measure.


In [ ]:
ranking = step6["ranking"]
feature_selection.show_table(ranking.head(35),
    note="Sorted by CombinedScore, ties broken by MI' and then by feature name, so "
         "the order is fully determined. Saved in full as combined_ranking.csv.")

### 6.6 The selected features

The top `N_SELECTED_FEATURES` by combined score. If the Pearson filter left fewer
than that, all of them are selected and a warning says so.


In [ ]:
selected = step6["selected"]
chosen = ranking[ranking["selected"]] if "selected" in ranking.columns \
    else ranking.head(len(selected))
feature_selection.show_table(
    chosen[["rank", "feature", "MI", "PI", "MI_norm", "PI_norm", "CombinedScore"]],
    note="The %d selected features, in rank order." % len(selected))

cut = float(chosen["CombinedScore"].iloc[-1])
rest = ranking[~ranking["feature"].isin(set(selected))]
if len(rest):
    print("lowest selected : %.4f  (%s)" % (cut, chosen["feature"].iloc[-1]))
    print("highest rejected: %.4f  (%s)"
          % (float(rest["CombinedScore"].iloc[0]), rest["feature"].iloc[0]))
    print("gap at the cut  : %.4f" % (cut - float(rest["CombinedScore"].iloc[0])))

### Top 30 by combined score

Each bar is split into its two parts, so a feature carried by MI alone and one
carried by PI alone are visibly different even though they land at the same
total. A bar that is almost entirely orange is a feature the forest leans on
that MI barely notices; an almost entirely blue one is the reverse - and that one
is often half of a correlated pair whose PI was suppressed.


In [ ]:
chart = config.STEP6_OUTPUT_DIR / config.STEP6_FILES["ranking_chart"]
funnel = config.STEP6_OUTPUT_DIR / config.STEP6_FILES["funnel"]
if feature_selection.dl.in_notebook():
    from IPython.display import Image, display
    for picture in (chart, funnel):
        if picture.exists():
            display(Image(filename=str(picture)))
        else:
            print("missing:", picture)
else:
    print("charts:", chart, funnel)

### Saved files

Everything below is in `C:\MSCAD2022\outputs\step6\`.

| file | what it is |
|---|---|
| `X_train_selected.csv` / `X_test_selected.csv` | the 30 selected columns, same order in both - **this is what Step 7 trains on** |
| `selected_features.json` | the ordered names plus every setting they were chosen under |
| `combined_ranking.csv` | every surviving feature with MI, PI, both normalised scores and `selected` |
| `mi_scores_all.csv` / `mi_scores.csv` | MI before and after the Pearson filter |
| `pearson_corr_matrix.csv`, `pearson_dropped_pairs.csv`, `pearson_remaining_pairs.csv` | the redundancy filter's working |
| `pi_scores.csv`, `rf_per_class_f1.csv`, `pi_duplicate_overlap.csv` | the permutation stage |
| `y_train.csv` / `y_test.csv` | copied from Step 5, unchanged |
| `rf_baseline.joblib` | the ranking forest - **not** a model for reporting |
| `step6_report.csv` | feature counts at each stage, and what was done |


In [ ]:
import pandas as pd

folder = config.usable_output_dir(config.STEP6_OUTPUT_DIR)
if folder is None:
    print("Step 6 outputs are not readable here:",
          config.why_unusable(config.STEP6_OUTPUT_DIR))
else:
    rows = []
    for key, name in config.STEP6_FILES.items():
        path = folder / name
        rows.append({"file": name, "exists": path.exists(),
                     "size (MB)": round(path.stat().st_size / 1024 / 1024, 3)
                                  if path.exists() else 0.0,
                     "what": key})
    feature_selection.show_table(pd.DataFrame(rows), note="Folder: %s" % folder)

    X_train_sel = pd.read_csv(folder / config.STEP6_FILES["X_train_selected"],
                              low_memory=False)
    X_test_sel = pd.read_csv(folder / config.STEP6_FILES["X_test_selected"],
                             low_memory=False)
    feature_selection.show_table(feature_selection.kv_table([
        ("X_train_selected", "%s rows x %d columns"
         % (f"{X_train_sel.shape[0]:,}", X_train_sel.shape[1])),
        ("X_test_selected", "%s rows x %d columns"
         % (f"{X_test_sel.shape[0]:,}", X_test_sel.shape[1])),
        ("identical columns, in order",
         list(X_train_sel.columns) == list(X_test_sel.columns)),
    ]))
    feature_selection.show_table(X_train_sel.head(5), all_columns=True,
                                 note="X_train_selected.head(5).")

### Verification

Every file was re-read from disk and checked: exactly N columns in the same order
in train and test, row counts still matching Step 5, labels aligned, no label or
identifier among the selected columns, no NaN or inf, and
`CombinedScore == MI_norm + PI_norm` in the saved ranking.


In [ ]:
if "saved" in step6 and "verification" in step6.get("saved", {}):
    feature_selection.show_table(step6["saved"]["verification"])
    print()
    print("ALL CHECKS PASSED:", step6["saved"]["ok"])
else:
    print("Step 6 was loaded from disk, so there is no fresh verification table.")
    print("Set OVERWRITE_STEP6 = True and re-run to select and verify again.")

### What Step 7 picks up

1. **Train on `X_train_selected.csv`.** `X_test_selected.csv` is touched once, at
   the end, to report a score. It already holds the same 30 columns in the same
   order, so no further alignment is needed.
2. **The imbalance is still untouched.** SMOTE, resampling or class weights go in
   Step 7 and must be fitted on the **training** side only. Oversampling before
   the split copies rows across it - the same leak as scaling before the split.
3. **The 30 features were chosen on the training rows alone**, so they carry no
   information from the test set. Re-running selection on the full dataset would
   quietly undo that.
4. **Report the Step 5.3 duplicate percentage next to every score**, and prefer
   macro-F1 and per-class recall over accuracy: with Brute_Force at tens of
   thousands of rows and Web_Crwling at 23, accuracy is close to a measure of the
   majority class alone.


---

## Step 7 - Autoencoder feature extraction

**Purpose** — Compress the 30 selected features to `AE_LATENT_DIM = 15` latent
features by training a network to reconstruct its own input.

**Input** — `C:\MSCAD2022\outputs\step6` — `X_train_selected.csv`,
`selected_features.json`. The test set is not read in this step.

**Method** — A symmetric autoencoder, 30 to 24 to 15 and back
(`AE_HIDDEN = [24]`), ReLU activations with a sigmoid output
(`AE_OUTPUT_ACT = "sigmoid"`), mean squared error loss, Adam with
`AE_LR = 0.001`, up to `AE_EPOCHS = 100` at `AE_BATCH_SIZE = 256`, early stopping
on an inner validation split (`AE_VAL_SIZE = 0.10`, `AE_PATIENCE = 10`) with the
best weights restored.

**Output** — `C:\MSCAD2022\outputs\step7` — `autoencoder.keras`,
`encoder.keras`, `ae_config.json`, the training history, the loss curve and the
per-feature reconstruction error.

**Why** — The autoencoder is unsupervised: its target is the input itself, so no
label is ever a target. Labels are used for one purpose only, to stratify the
inner validation split so that the smallest classes are represented in it. A
sigmoid output is used because Step 5 scaled every training feature into
$[0, 1]$, which is exactly the range a sigmoid can produce.

Reconstruction loss over $n$ samples and $d$ features:

$$\mathcal{L} = \frac{1}{n\,d}\sum_{i=1}^{n}\sum_{j=1}^{d}\bigl(x_{ij} - \hat{x}_{ij}\bigr)^{2}$$

Only the encoder half is retained for Step 8; the decoder exists to create the
training signal and is then discarded.

An autoencoder is a network trained to rebuild **its own input** through a
narrow middle layer. Squeeze 30 features through 15 numbers and force the
network to reconstruct the 30, and the 15 have to carry whatever structure the
30 share. Those 15 are the new feature set for Steps 10-12.

```
    30 features  ->  24  ->  15 (latent)  ->  24  ->  30 reconstruction
                 \___ encoder ___/        \___ decoder ___/
```

Only the **encoder** half is kept for later use. The decoder exists to create
the training signal and is then discarded.

**It is unsupervised.** The loss is the reconstruction error against the input
itself, so no label is ever a target. Labels appear in Step 7 for exactly one
purpose: to *stratify* the inner split used for early stopping, so the 23-row
classes are represented in the validation part instead of landing entirely in
the fit part by chance. Nothing about the encoding depends on them.

**Why a sigmoid output.** Step 5's MinMax scaler put every training feature into
`[0, 1]`, so a sigmoid can reach every legal value and nothing else. A linear
output would be free to predict -3 or 7 - values no input can take - and would
waste capacity learning not to.

> **Framework: TensorFlow/Keras, not PyTorch.** The rule was "PyTorch unless
> TensorFlow/Keras officially supports Windows + Python 3.13", and it does:
> TensorFlow 2.21.0 publishes `tensorflow-2.21.0-cp313-cp313-win_amd64.whl` on
> PyPI, so `pip install tensorflow` works on this machine with no source build.
> That wheel is CPU-only - native-Windows GPU support ended after TF 2.10 -
> which is all these steps need. The same framework is used for Steps 10-12, so
> there is one set of seeds, one save format and one set of versions to cite.
>
> ```
> "C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m pip install "tensorflow>=2.21,<3"
> ```

**Reproducibility.** `set_all_seeds()` seeds Python, NumPy and Keras from
`RANDOM_STATE` and calls `tf.config.experimental.enable_op_determinism()`.
Without that last call the same seed still drifts run to run, because several
CPU kernels reduce in a non-deterministic order. With it, two runs give
bit-identical codes - there is a test for exactly that.


### Step 7 settings

In [ ]:
from src.mscad import autoencoder

versions = autoencoder.framework_versions()
autoencoder.show_table(autoencoder.kv_table([
    ("input folder", str(config.STEP6_OUTPUT_DIR)),
    ("Step 7 output", str(config.STEP7_OUTPUT_DIR)),
    ("Step 8 output", str(config.STEP8_OUTPUT_DIR)),
    ("architecture", "%d -> %s -> %d -> %s -> %d"
     % (config.N_SELECTED_FEATURES, config.AE_HIDDEN, config.AE_LATENT_DIM,
        config.AE_HIDDEN, config.N_SELECTED_FEATURES)),
    ("activation / output", "%s / %s" % (config.AE_ACTIVATION, config.AE_OUTPUT_ACT)),
    ("loss", config.AE_LOSS),
    ("epochs / batch / lr", "%d / %d / %s"
     % (config.AE_EPOCHS, config.AE_BATCH_SIZE, config.AE_LR)),
    ("early stopping", "val_size %s, patience %d, best weights restored"
     % (config.AE_VAL_SIZE, config.AE_PATIENCE)),
    ("framework", "%s %s, keras %s"
     % (versions["framework"], versions["tensorflow"], versions["keras"])),
    ("python / numpy", "%s / %s" % (versions["python"], versions["numpy"])),
    ("OVERWRITE_STEP7 / STEP8", "%s / %s"
     % (config.OVERWRITE_STEP7, config.OVERWRITE_STEP8)),
    ("outputs already exist", "%s / %s"
     % (autoencoder.step7_outputs_exist(), autoencoder.step8_outputs_exist())),
]), note="These versions are written into ae_config.json, so a result can be "
         "reproduced from the file alone.")

### Run Step 7

Training is the slow part. On a CPU with ~84,000 training rows, 100 epochs at
batch 256 takes a few minutes; early stopping usually ends it sooner. The
elapsed time and the epoch it stopped at are both reported.


In [ ]:
step7 = autoencoder.run_step7()

### 7.1 Architecture

`units` is the width of each layer and `parameters` is how many weights it
holds. The decoder mirrors the encoder's **shape**, not its values - the two
halves have separate weights, which is why the parameter count is roughly
symmetric rather than halved.


In [ ]:
if step7.get("status") != "loaded_existing":
    autoencoder.show_table(step7["architecture"],
        note="Only the encoder half (input through `latent`) is used in Step 8.")
else:
    autoencoder.show_table(step7["architecture"],
                           note="Read back from architecture.csv.")

### 7.2 Training

The numbers below are **reconstruction errors**, not classification scores: the
target is the input itself. A validation MSE close to the training MSE means the
bottleneck is genuinely compressing rather than memorising.


In [ ]:
info = step7["info"]
autoencoder.show_table(autoencoder.kv_table([
    ("best epoch", info.get("best_epoch")),
    ("epochs run", info.get("epochs_run")),
    ("train reconstruction MSE", "%.6f" % info["train_mse"]),
    ("validation reconstruction MSE", "%.6f" % info["val_mse"]),
    ("training time", "%.1f s" % info["seconds"] if info.get("seconds") else "-"),
]), note="Early stopping restored the weights from the best epoch, so these are "
         "the weights that were saved - not the last epoch's.")

curve = config.STEP7_OUTPUT_DIR / config.STEP7_FILES["loss_curve"]
if curve.exists() and autoencoder.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(curve)))
elif not curve.exists():
    print("No loss curve at", curve)

### Per-feature reconstruction error

Which of the 30 features the 15-number bottleneck could **not** keep. A high
value means that feature is either noisy or carries something none of the others
do - worth knowing before Steps 10-12, because whatever the autoencoder threw
away is not available to any classifier downstream.


In [ ]:
table = step7.get("per_feature_mse")
if table is not None:
    autoencoder.show_table(table,
        note="On the inner validation part. Saved as per_feature_mse.csv.")
else:
    print("Not available on the loaded path - set OVERWRITE_STEP7 = True to recompute.")

---

## Step 8 - Transform the training and test sets

**Purpose** — Apply the frozen encoder to both the training and the test
features.

**Input** — `C:\MSCAD2022\outputs\step6` (the selected features) and
`C:\MSCAD2022\outputs\step7\encoder.keras`.

**Method** — $Z_{\text{train}} = \text{encoder}(X_{\text{train}})$ and
$Z_{\text{test}} = \text{encoder}(X_{\text{test}})$, producing columns `z01` to
`z15` in the original row order. `encode()` verifies the feature names and their
order before every call.

**Output** — `C:\MSCAD2022\outputs\step8` — `Z_train.csv`, `Z_test.csv`, the
label files copied unchanged, and the reconstruction error for reference.

**Why** — The test features are read here for the first time in Steps 7-8, and
only to pass through an encoder that is already trained. A `Dense` layer reads
column positions rather than names, so a frame with the correct columns in the
wrong order would produce plausible but incorrect codes; the column check makes
that impossible.

The 15 latent columns have no individual interpretation: `z07` is one axis of a
learned compression, not a named feature. Feature-importance statements about the
original features come from the Step 6 ranking, not from the latent columns.

The **same frozen encoder** runs over both sets:

```
    Z_train = encoder(X_train_selected)      Z_test = encoder(X_test_selected)
```

`X_test_selected` is read here for the first time in Steps 7-8, and only to be
pushed through an encoder that is already trained and unchanged. Nothing about
the architecture, the epochs or the early stopping came from it.

**Row order is unchanged**, so row *i* of `Z_train` is still row *i* of
`X_train_selected`, and `y_train` still belongs to it.

> **`encode()` checks the columns before every call.** A `Dense` layer reads
> **positions**, not names - hand it the same 30 columns in a different order and
> it returns numbers that look perfectly reasonable and are wrong. The saved
> encoder therefore ships with the feature order it was trained on, and `encode()`
> refuses any frame that does not match it exactly, naming the first position that
> differs.


In [ ]:
step8 = autoencoder.run_step8(
    encoder=step7.get("encoder"),
    autoencoder=step7.get("autoencoder"),
    feature_order=step7.get("feature_order"))

### The encoded matrices

In [ ]:
import pandas as pd

Z_train = step8["Z_train"]
Z_test = step8["Z_test"]
autoencoder.show_table(autoencoder.kv_table([
    ("Z_train", "%s rows x %d columns" % (f"{Z_train.shape[0]:,}", Z_train.shape[1])),
    ("Z_test", "%s rows x %d columns" % (f"{Z_test.shape[0]:,}", Z_test.shape[1])),
    ("columns", "%s ... %s" % (Z_train.columns[0], Z_train.columns[-1])),
    ("compression", "%d features -> %d numbers"
     % (config.N_SELECTED_FEATURES, config.AE_LATENT_DIM)),
]))
autoencoder.show_table(Z_train.head(5), all_columns=True,
                       note="Z_train.head(5). These are the latent codes, not the "
                            "original features - they have no individual meaning.")

if "recon_mse" in step8:
    autoencoder.show_table(step8["recon_mse"],
        note="**Reported only.** No decision in Steps 7-8 was made by looking at "
             "the test error. A test MSE close to the train MSE means the encoder "
             "generalises; a much larger one would mean it had memorised the "
             "training rows.")

### Verification

Re-read from disk: `Z_train` is `(N_train, 15)`, `Z_test` is `(N_test, 15)`, both
carry `z01 ... z15` in the same order, no NaN or inf, and the labels are still
aligned row for row.


In [ ]:
if "verification" in step8:
    autoencoder.show_table(step8["verification"])
    print()
    print("ALL CHECKS PASSED:", step8["ok"])
else:
    print("Step 8 was loaded from disk, so there is no fresh verification table.")
    print("Set OVERWRITE_STEP8 = True and re-run to encode and verify again.")

### Re-using the encoder later

The encoder is saved as a Keras model plus the feature order it expects, so any
later step can load it without touching Step 7:

```python
from src.mscad import autoencoder
encode, model, feature_order = autoencoder.load_encoder()
Z = encode(X)          # raises unless X has exactly those columns, in that order
```


### What Step 9 picks up

1. **Train on `Z_train.csv`** (15 columns) with `y_train.csv`. `Z_test.csv` is
   touched once, at the end, to report a score.
2. **The 15 latent columns have no individual meaning.** `z07` is not a feature
   anyone can name - it is one axis of a learned compression. Feature-importance
   claims about `z` columns do not transfer back to the original 30; use the
   Step 6 ranking for that.
3. **The imbalance is still untouched.** It has been left alone deliberately
   through Steps 5-8. SMOTE, resampling or class weights belong in a later step
   and must be fitted on the **training** side only.
4. **Whatever the autoencoder discarded is gone** for every classifier
   downstream. The per-feature reconstruction table in 7.2 is the record of what
   that was.


---

## Step 9 - Class imbalance handled in the training loss

**Purpose** — Compensate for class imbalance by weighting the training loss, with
no resampling of any kind.

**Input** — `C:\MSCAD2022\outputs\step8` — `Z_train.csv`, `y_train.csv`.

**Method** — A stratified inner split of the training set
(`CLF_VAL_SIZE = 0.10`), then balanced class weights computed from the fit part
only (`IMBALANCE_STRATEGY = "class_weight"`,
`CLASS_WEIGHT_SCHEME = "balanced"`), normalised so that the mean per-sample
weight is 1.

**Output** — `C:\MSCAD2022\outputs\step9` — `Z_fit.csv`, `Z_val.csv`, the
matching label files, `fit_sample_weights.csv` and `class_weights.json`.

**Why** — No SMOTE, no SMOTE-ENN and no over- or undersampling are used. Roughly
half of the rows in this dataset share identical feature values once identifiers
are removed, so interpolating between nearest neighbours would duplicate existing
rows rather than add information. Weighting changes how much a row counts in the
loss and never changes how many rows exist. The weights are applied to the
training loss only: the validation loss is unweighted and the test set is never
reweighted.

Balanced class weight for class $c$, with $n$ samples and $k$ classes:

$$w_c = \frac{n}{k \cdot n_c}$$

Under this definition $n_c \cdot w_c$ is identical for every class, so each class
contributes equally to the loss.

The imbalance has been left alone deliberately since Step 4. This is where it is
addressed - **by class weighting only**.

| | |
|---|---|
| **what it does** | makes a rare class's rows count more in the training loss |
| **what it does not do** | add, remove or duplicate a single row |
| **where it applies** | the training loss, and nowhere else |
| **what it never touches** | the test set |

**No resampling of any kind** - no SMOTE, no SMOTE-ENN, no over- or
undersampling - and `imbalanced-learn` is deliberately not a dependency. That is
not squeamishness about SMOTE in general; it is about this dataset in particular.
Around half of these rows share identical feature values once the identifiers are
gone, because thousands of scan, flood and brute-force flows genuinely look
alike. SMOTE interpolates between nearest neighbours, so *between identical rows
it manufactures exact copies* rather than new information - and the Step 5.3
leakage figure would get worse, not better. Weighting adds no rows at all.

**The test set is never reweighted or resampled.** Weights come from the training
rows alone and apply only to the training loss; the test set is scored exactly as
Step 8 produced it. Rebalancing a test set quietly changes the question: a
macro-F1 on a reweighted test set is not the number a reader will assume it is.
Step 9 fingerprints `Z_test.csv` and `y_test.csv` with SHA-256 before and after,
so "unchanged" is a fact rather than an assumption.

**Step 9 trains nothing.** It produces the fit/validation split and the weights
that Steps 10-12 reuse for every model configuration:

```python
X_fit, y_fit, X_val, y_val, weights, sample_w = imbalance.make_training_sets(
    Z_train, y_train, strategy=config.IMBALANCE_STRATEGY)

model.fit(X_fit, y_fit,
          validation_data=(X_val, y_val),           # UNWEIGHTED on purpose
          class_weight=imbalance.keras_class_weight(weights))
```

The validation loss used for early stopping is deliberately **unweighted**: it
has to measure the model on the data as it really is, not on the distribution the
training loss pretends to see.


### Step 9 settings

In [ ]:
from src.mscad import imbalance

imbalance.show_table(imbalance.kv_table([
    ("input folder", str(config.STEP8_OUTPUT_DIR)),
    ("output folder", str(config.STEP9_OUTPUT_DIR)),
    ("IMBALANCE_STRATEGY", config.IMBALANCE_STRATEGY),
    ("CLASS_WEIGHT_SCHEME", config.CLASS_WEIGHT_SCHEME),
    ("CLASS_WEIGHT_MAX", config.CLASS_WEIGHT_MAX),
    ("NORMALISE_WEIGHTS", config.NORMALISE_WEIGHTS),
    ("CLF_VAL_SIZE", config.CLF_VAL_SIZE),
    ("RANDOM_STATE", config.RANDOM_STATE),
    ("OVERWRITE_STEP9", config.OVERWRITE_STEP9),
    ("outputs already exist", imbalance.step9_outputs_exist()),
]), note="`balanced` is sklearn's w_c = n / (k x n_c). `sqrt_balanced` is its "
         "square root -- milder, and worth comparing if training turns out "
         "unstable. `CLASS_WEIGHT_MAX` caps any single weight.")

### Run Step 9

In [ ]:
step9 = imbalance.run_step9()

### 9.1 The weights

`effective weight share %` is `(fit rows x weight)` as a percentage of the total.
With **balanced** weights every class should land on the same value - that flat
column *is* what balancing means, and seeing it come out flat is the check that
the weights do what they claim.


In [ ]:
table = step9.get("weight_table")
if table is not None:
    imbalance.show_table(table,
        note="`class weight` is what multiplies that class's loss. `share of fit %` "
             "is what the data actually looks like; `effective weight share %` is "
             "what the loss sees.")
else:
    print("Not available on the loaded path - set OVERWRITE_STEP9 = True.")

### How extreme are these weights?

> **With `balanced` on the real MSCAD training set, the largest and smallest
> weights differ by roughly 1000x or more** - around 88,000 `Brute_Force` rows
> against about 18 training rows for `Normal` and `Web_Crwling`. That is the
> honest consequence of asking every class to count equally, but a weight of
> ~1000 on 18 rows means a handful of examples can dominate a gradient step, and
> training can oscillate or diverge.
>
> If that happens, two milder settings are already wired in and worth comparing
> rather than guessing:
>
> | setting | effect |
> |---|---|
> | `CLASS_WEIGHT_SCHEME = "sqrt_balanced"` | ~1000x becomes ~32x |
> | `CLASS_WEIGHT_MAX = 100.0` | nothing exceeds 100x, the rest unchanged |
>
> **The default stays `"balanced"`.** Change it only on evidence - an unstable
> loss curve in Step 10 - and report which setting produced the numbers you
> quote.


In [ ]:
stats = step9.get("stats")
if stats is not None:
    imbalance.show_table(imbalance.kv_table([
        ("largest weight", "%.4f" % stats["max_weight"]),
        ("smallest weight", "%.4f" % stats["min_weight"]),
        ("ratio", "%.1fx" % stats["ratio"]),
        ("effective share, even split", "%.4f %%" % stats["even_share"]),
        ("worst deviation from even", "%.6f pp" % stats["spread_pp"]),
    ]))
    if stats["ratio"] >= 1000:
        print("[NOTE] the weights span %.0fx. Watch the Step 10 loss curve for"
              % stats["ratio"])
        print("       instability; sqrt_balanced or CLASS_WEIGHT_MAX are the"
              " fallbacks.")

### Charts

Both on a log scale, because the counts and the weights each span three orders of
magnitude - and because a log axis makes exact values hard to read, every bar
carries its own number.


In [ ]:
for key in ("fit_counts_chart", "weights_chart"):
    picture = config.STEP9_OUTPUT_DIR / config.STEP9_FILES[key]
    if picture.exists() and imbalance.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(picture)))
    elif not picture.exists():
        print("missing:", picture)

### 9.2 Checks

The things that would silently invalidate every later result if they were wrong:
the two parts add up to the training set with nothing added or lost, every class
reaches both parts, the weights came from the fit part alone, and the test files
are byte-for-byte what Step 8 wrote.


In [ ]:
if "verification" in step9:
    imbalance.show_table(step9["verification"])
    print()
    print("ALL CHECKS PASSED:", step9["ok"])
else:
    print("Step 9 was loaded from disk, so there is no fresh verification table.")
    print("Set OVERWRITE_STEP9 = True and re-run to recompute and verify.")

### Saved files

In `C:\MSCAD2022\outputs\step9\`:

| file | what it is |
|---|---|
| `Z_fit.csv` / `y_fit.csv` | what Steps 10-12 train on |
| `Z_val.csv` / `y_val.csv` | the inner validation part, for early stopping - **unweighted** |
| `fit_sample_weights.csv` | one weight per fit row, for a framework that wants per-sample weights |
| `class_weights.json` | per class: label, encoded label, fit rows, weight, plus the scheme, cap and normalisation flag |
| `imbalance_report.csv` | what was done, in order |


In [ ]:
import pandas as pd

folder = config.usable_output_dir(config.STEP9_OUTPUT_DIR)
if folder is None:
    print("Step 9 outputs are not readable here:",
          config.why_unusable(config.STEP9_OUTPUT_DIR))
else:
    rows = []
    for key, name in config.STEP9_FILES.items():
        path = folder / name
        rows.append({"file": name, "exists": path.exists(),
                     "size (MB)": round(path.stat().st_size / 1024 / 1024, 3)
                                  if path.exists() else 0.0,
                     "what": key})
    imbalance.show_table(pd.DataFrame(rows), note="Folder: %s" % folder)

    weights = step9["class_weights"]
    print("keras class_weight dict for Steps 10-12:")
    print({k: round(v, 4) for k, v in sorted(weights.items())})

### What Step 10 picks up

1. **Train on `Z_fit.csv` / `y_fit.csv`** with `class_weight=` from
   `class_weights.json`. Validate on `Z_val.csv` / `y_val.csv`
   **without** weights, and early-stop on validation macro-F1.
2. **The test set stays untouched** until the very end, and is never reweighted.
3. **Report which weighting produced each number.** A macro-F1 under
   `balanced` and one under `none` are different experiments, and the comparison
   between them is itself a result worth having in the thesis.
4. **If the loss oscillates**, try `sqrt_balanced` or `CLASS_WEIGHT_MAX` before
   touching the architecture - a 1000x weight ratio is the more likely culprit.


---

## Step 10 - The proposed hybrid CNN-LSTM

**Purpose** — Train the proposed classifier on the latent features.

**Input** — `C:\MSCAD2022\outputs\step8` (the encoded training set) and the
Step 9 split and class weights.

**Method** — `Conv1D(64, k=3)` to `BatchNormalization` to `Conv1D(64)` to
`MaxPooling1D(2)` to `Dropout(0.3)` to `LSTM(64)` to `Dropout(0.3)` to
`Dense(64)` to `Dense(n_classes)`. Adam with `CLF_LR = 0.001`, up to
`CLF_EPOCHS = 50` at `CLF_BATCH_SIZE = 256`, early stopping on
`EARLY_STOP_METRIC = "val_macro_f1"` with `CLF_PATIENCE = 8` and the best weights
restored.

**Output** — `C:\MSCAD2022\outputs\step10` — `cnn_lstm.keras`,
`model_card.json`, the training history and the loss and macro-F1 curves.

**Why** — The output layer emits logits with no activation and the loss applies
the softmax internally, so the softmax is applied exactly once. Early stopping
tracks macro-F1 rather than loss because the loss is dominated by the majority
class, and a model that stops predicting the smallest classes can still reduce
it. The test set is not read during training, early stopping, tuning or
weighting.

> **Input arrangement.** With `INPUT_MODE = "feature_seq"` each flow is arranged
> as a tensor of shape $(15, 1)$: the 15 latent features as 15 steps of one
> channel. The convolutional layers learn patterns across neighbouring latent
> features and the LSTM reads the resulting feature maps as a sequence of learned
> feature patterns. **These are feature maps, not time steps**, and the latent
> units have no natural ordering, so results obtained in this mode are not
> evidence about temporal attack structure. The alternative
> `INPUT_MODE = "time_window"` builds windows of consecutive flows within one
> capture and is the mode required for any temporal claim.

The full proposed pipeline, end to end:

```
66 features -> Step 6 selection -> 30 -> Step 7 autoencoder -> 15 -> CNN-LSTM -> 6 classes
```

Architecture: `Conv1D(64,k3) -> BatchNorm -> Conv1D(64) -> MaxPool(2) -> Dropout
-> LSTM(64) -> Dropout -> Dense(64) -> Dense(n_classes)`.

> ### What the input actually is - read this before quoting a result
>
> `INPUT_MODE = "feature_seq"` lays each flow's 15 autoencoder features out as a
> tensor of shape `(15, 1)`: fifteen "steps" of one channel. The Conv1D layers
> learn local patterns across **neighbouring latent features**, and the LSTM
> reads their output feature map as a sequence of **learned feature patterns**.
>
> **This is not a time sequence, and the model is not doing temporal
> modelling.** Worse, the autoencoder's latent units have *no natural order* -
> which feature lands at position 3 rather than position 11 is an arbitrary
> consequence of how the encoder initialised, so "neighbouring" means nothing in
> particular. The arrangement is fixed and reproducible, because the encoder is
> saved, and this is the architecture the thesis proposes, so it is implemented
> as specified. But a claim that the CNN-LSTM captures temporal attack structure
> needs `INPUT_MODE = "time_window"`, not this one. The same caveat is written
> into `model_card.json`, so it travels with the model.
>
> `INPUT_MODE = "time_window"` builds windows of `WINDOW_SIZE` consecutive flows
> ordered by `Timestamp` **within one capture**, labelled by the last flow, built
> separately on each side of the split so no window ever straddles it. Its own
> caveat: the Step 5 split was random and stratified, so consecutive rows on one
> side were not consecutive in the capture - there are gaps, and a window spans
> more wall-clock time than it appears to.

**Three details that are easy to get wrong**, and how they are handled:

1. **The output layer emits logits**, with no activation, and the loss is
   `SparseCategoricalCrossentropy(from_logits=True)`, which applies the softmax
   internally. Softmax is applied exactly once, in `predict_proba()`. Doing both
   would apply it twice - the gradients would be wrong and the model would train
   badly while still producing plausible-looking probabilities.
2. **The validation loss is unweighted.** Keras applies `class_weight` to
   training batches only, which is what we want: the validation signal has to
   measure the model on the data as it really is.
3. **Early stopping watches macro-F1, not loss.** With a 3849:1 imbalance the
   loss is dominated by the majority class, and a model that quietly stops
   predicting the 23-row classes can still improve it.

The test set is not read anywhere in Step 10 - not for training, early stopping,
tuning or the class weights.


### 10.1 Approach 1: Hybrid CNN-LSTM

The cells below are Approach 1, unchanged: the model, the training run and the
curves this project has reported from the start. Nothing in this section is
re-run or retrained by Approach 2, and its saved outputs in
`C:\MSCAD2022\outputs\step10` are never rewritten.

Approach 2 follows in 10.2 and shares everything here except one block.

### Step 10 settings

In [ ]:
from src.mscad import models

models.show_table(models.kv_table([
    ("INPUT_MODE", config.INPUT_MODE),
    ("WINDOW_SIZE (time_window only)", config.WINDOW_SIZE),
    ("CNN filters / kernel / pool", "%s / %d / %d"
     % (config.CNN_FILTERS, config.CNN_KERNEL, config.CNN_POOL)),
    ("LSTM units", config.LSTM_UNITS),
    ("dropout / dense units", "%s / %d" % (config.DROPOUT, config.DENSE_UNITS)),
    ("epochs / batch / lr", "%d / %d / %s"
     % (config.CLF_EPOCHS, config.CLF_BATCH_SIZE, config.CLF_LR)),
    ("early stopping", "%s, patience %d, best weights restored"
     % (config.EARLY_STOP_METRIC, config.CLF_PATIENCE)),
    ("imbalance", "%s / %s (Step 9)"
     % (config.IMBALANCE_STRATEGY, config.CLASS_WEIGHT_SCHEME)),
    ("N_SEEDS", config.N_SEEDS),
    ("OVERWRITE_STEP10 / STEP11", "%s / %s"
     % (config.OVERWRITE_STEP10, config.OVERWRITE_STEP11)),
]), note="`N_SEEDS > 1` retrains with different seeds and reports mean +- std, "
         "which the rare classes need: with ~5 test rows each, one row changing "
         "hands moves a per-class F1 a long way.")

### Run Step 10

Training is the slow part. Early stopping usually ends it well before
`CLF_EPOCHS`; the elapsed time and the epoch it stopped at are both reported.


In [ ]:
step10 = models.run_step10()

### Architecture

In [ ]:
models.show_table(step10["architecture"],
    note="The last layer's activation is `linear` -- it emits LOGITS. The loss "
         "applies the softmax; `predict_proba()` is the only other place one is "
         "applied.")

### Training curves

The loss curve carries two series that are deliberately **not** comparable: the
training loss is class-weighted and the validation loss is not. Watch their
shapes, not the gap between them. The macro-F1 curve is the one early stopping
acts on.


In [ ]:
for key in ("loss_curve", "f1_curve"):
    picture = config.STEP10_OUTPUT_DIR / config.STEP10_FILES[key]
    if picture.exists() and models.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(picture)))
    elif not picture.exists():
        print("missing:", picture)

---

### 10.2 Approach 2: Hybrid CNN-Attention-LSTM

**Purpose** — Ask whether letting the model weigh parts of the convolved
latent vector, instead of taking the LSTM's last state, buys anything.

**Input** — The same Step 8 matrices Approach 1 used: `Z_train` (15
autoencoder features), with `INPUT_MODE = "feature_seq"`, so each flow is
reshaped to (15, 1).

**Method** — Approach 1's stack with **one** block added:

```
Conv1D(64, 3) -> BatchNorm -> Conv1D(64, 3) -> MaxPool(2) -> Dropout
  -> LSTM(64, return_sequences=True)
  -> additive attention            <- the only difference
  -> Dropout -> Dense(64) -> Dense(n_classes)   [logits]
```

**Output** — `C:\MSCAD2022\outputs\step10_attention` — the model, the
training history, both curves, the architecture table and `model_card.json`.

**Why** — Approach 1's LSTM compresses the whole sequence into its final
state, so an early position can only reach the classifier through everything
that follows it. Attention gives the classifier a direct, weighted path to
every position, and the weights are readable afterwards.

#### The attention, written out

Given the LSTM output sequence $h_1 \dots h_T$:

$$e_t = v^{\top} \tanh\!\left(W h_t + b\right)
\qquad
\alpha_t = \frac{\exp(e_t)}{\sum_{s=1}^{T}\exp(e_s)}
\qquad
c = \sum_{t=1}^{T} \alpha_t\, h_t$$

$W \in \mathbb{R}^{64 \times 64}$ maps each LSTM state into the scoring space
(`ATTN_UNITS`), $v$ scores it, the softmax turns the scores into weights that
are non-negative and sum to one, and $c$ is the weighted summary the dense head
sees. This is additive (Bahdanau-style) attention; it adds about four thousand
parameters to the fifty thousand Approach 1 already has.

| | Approach 1 | Approach 2 |
|---|---|---|
| CNN front end | Conv -> BN -> Conv -> Pool -> Dropout | identical |
| LSTM | 64 units, last state | 64 units, **full sequence** |
| pooling | — | **additive attention** |
| head | Dropout -> Dense(64) -> Dense(C) | identical |
| everything else | — | identical: epochs, batch, lr, patience, seed, class weights, split |

> **What the attention positions are.** With `INPUT_MODE = "feature_seq"` a
> flow is one vector, not a sequence of events, so after the convolutions and
> pooling the model sees about seven **CNN feature-map positions**. The
> attention says which part of the convolved latent vector it leaned on. These
> are **not time steps**, and reading them as "which moment of the attack"
> would be wrong. Temporal attention would need
> `INPUT_MODE = "time_window"`.

> **Same protocol, asserted.** The Step 9 fit/validation split, the class
> weights on the training loss only, the unweighted validation loss, early
> stopping on validation macro-F1 with the best weights restored, and the same
> seed. The checks below compare these against Approach 1's own saved
> `model_card.json` and Step 9's `class_weights.json`.

In [ ]:
# Quieten TensorFlow's C++ logger for this section only.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from src.mscad import models_attention

step10b = models_attention.run_step10b()

---

## Step 11 - Evaluation on the untouched test set

**Purpose** — Measure the trained model on the test set, which has not been used
for any earlier decision.

**Input** — `C:\MSCAD2022\outputs\step8` (`Z_test.csv`, `y_test.csv`),
`C:\MSCAD2022\outputs\step10` (the trained model), and the Step 5 feature
matrices for the duplicate check.

**Method** — Softmax of the model's logits, then accuracy, macro and weighted
precision, recall and F1, G-mean, one-vs-rest ROC-AUC, the per-class report, both
confusion matrices, a benign-versus-attack binary view, and CPU inference timing
measured after a warm-up batch.

**Output** — `C:\MSCAD2022\outputs\step11` — `metrics.json`,
`per_class_report.csv`, `predictions.csv`, the confusion matrices and the
duplicate-free comparison.

**Why** — Every headline metric is reported twice: over all test rows, and over
only those test rows that have no feature-identical row in the training set.
Approximately 54% of this dataset's rows share identical feature values once
identifiers are removed, so a random split places identical rows on both sides
and the full-test score is optimistic. The duplicate-free score is the
conservative estimate of generalisation. The same model and the same predictions
are used for both; only the set of scored rows differs.

Macro-F1 over $k$ classes, and G-mean over per-class recall:

$$\text{macro-F1} = \frac{1}{k}\sum_{c=1}^{k} F1_c, \qquad
\text{G-mean} = \left(\prod_{c=1}^{k} \text{recall}_c\right)^{1/k}$$

G-mean becomes zero if any single class is never recalled, which macro-F1 and
accuracy can both conceal.

> **Rare classes.** The smallest classes contribute only about five test rows
> each. Their per-class precision, recall and F1 change substantially when a
> single row is classified differently, so they should be read as indicative.
> Setting `N_SEEDS` to 3 or 5 reports mean and standard deviation across seeds.

This is the first and only time the test set is used. It was never resampled,
reweighted, or consulted for training, early stopping, tuning or the class
weights.

### Why there are two scores for everything

About half this dataset's rows share identical feature values once the
identifiers are gone. A random split therefore puts feature-identical twins on
both sides, and the model can "recognise" a test row it has effectively already
seen. Every headline metric is reported twice:

| | |
|---|---|
| **full** | every test row |
| **duplicate-free** | only test rows with no feature-identical row in training |

Nothing is retrained and nothing is dropped from the model's input - the same
predictions are simply scored over two row sets. **The duplicate-free number is
the conservative estimate of generalisation, and is the one to lead with.**

### Which metric to quote

Accuracy on this dataset is close to a measure of the majority class alone:
`Brute_Force` is ~84% of the rows, so predicting it for everything scores ~84%.

- **macro-F1** weights every class equally
- **G-mean** is the geometric mean of per-class recall, and goes to **zero** the
  moment any class is never recalled - the strictest of the three, and the one
  that catches a model quietly abandoning a rare class

> **The rare classes have about five test rows each.** Their per-class
> precision, recall and F1 move a long way when a single row changes hands, so
> treat them as indicative, not precise. `N_SEEDS = 3` or `5` and a reported
> mean +- std is the honest way to quote them.


In [ ]:
from src.mscad import evaluation

step11 = evaluation.run_step11(
    model=step10.get("model"),
    training_seconds=step10.get("info", {}).get("seconds"))

### Full vs duplicate-free

In [ ]:
comparison = step11.get("duplicate_comparison")
if comparison is not None:
    evaluation.show_table(comparison,
        note="Same model, same predictions -- only the row set differs.")
    evaluation.show_table(step11["duplicate_counts"],
        note="Per class: how many test rows, and how many of them have a "
             "feature-identical twin in training. A class with a HIGH duplicated "
             "percentage is one the model may be recognising rather than "
             "generalising to.")
else:
    evaluation.show_table(step11["duplicate_split"],
                          note="Read back from duplicate_free_comparison.csv.")

### Per class

In [ ]:
table = step11.get("per_class")
if table is not None:
    evaluation.show_table(table,
        note="`support` is the number of test rows. Where it is small, the other "
             "three columns are high-variance -- read them with that in mind.")

### Confusion matrix

Rows are the true class and are normalised to 1, so each row shows where that
class's flows actually went. The raw count is in brackets.


In [ ]:
picture = config.STEP11_OUTPUT_DIR / config.STEP11_FILES["confusion_chart"]
if picture.exists() and evaluation.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(picture)))
elif not picture.exists():
    print("missing:", picture)

counts = step11.get("confusion_counts")
if counts is not None:
    evaluation.show_table(counts.reset_index(), all_columns=True,
                          note="Raw counts. Rows are truth, columns are predictions.")

### Binary view - benign vs attack

Derived from the same multiclass probabilities:
`P(attack) = 1 - sum(benign class probabilities)`. The **false-positive rate** is
the operational number here: it is the share of benign flows an analyst would be
paged about.


In [ ]:
binary = step11.get("binary")
if isinstance(binary, dict):
    evaluation.show_table(evaluation.kv_table([
        ("accuracy", "%.4f" % binary["accuracy"]),
        ("precision", "%.4f" % binary["precision"]),
        ("recall (attacks caught)", "%.4f" % binary["recall"]),
        ("F1", "%.4f" % binary["f1"]),
        ("ROC-AUC", evaluation._fmt(binary["roc_auc"])),
        ("false-positive rate", "%.4f" % binary["false_positive_rate"]),
        ("TN / FP / FN / TP", "%d / %d / %d / %d"
         % (binary["true_negatives"], binary["false_positives"],
            binary["false_negatives"], binary["true_positives"])),
    ]), note="A benign flow called an attack is a false positive. With very few "
             "benign test rows the FPR is itself a high-variance number.")
elif binary is not None:
    evaluation.show_table(binary)

### Timing and the G-mean cross-check

In [ ]:
timing = step11.get("inference")
if timing is not None:
    evaluation.show_table(evaluation.kv_table([
        ("training time (Step 10)", "%.1f s" % step11["training_seconds"]
         if step11.get("training_seconds") else "-"),
        ("inference rows", f"{timing['rows']:,}"),
        ("inference total", "%.3f s (+- %.3f over %d runs)"
         % (timing["total_seconds"], timing["std_seconds"], timing["runs"])),
        ("per 1,000 flows", "%.1f ms" % timing["ms_per_1000_flows"]),
        ("batch size", timing["batch_size"]),
    ]), note="CPU wall clock, measured after a warm-up batch -- the first call "
             "builds the graph, so timing it would measure compilation.")

check = step11.get("gmean_cross_check")
if check is not None:
    print("G-mean (computed directly): %.6f" % check["ours"])
    if check["imblearn"] is None:
        print(check["note"])
    else:
        print("G-mean (imblearn)         : %.6f" % check["imblearn"])
        print("agree:", check["agree"])

### Saved files

In `C:\MSCAD2022\outputs\step11\`: `metrics.json`, `per_class_report.csv`,
`predictions.csv` (`Orig_Row`, truth, prediction, every class probability and
`has_train_duplicate`), both confusion matrices, `duplicate_free_comparison.csv`,
`binary_view.csv` and `step11_report.csv`.


In [ ]:
import pandas as pd

folder = config.usable_output_dir(config.STEP11_OUTPUT_DIR)
if folder is None:
    print("Step 11 outputs are not readable here:",
          config.why_unusable(config.STEP11_OUTPUT_DIR))
else:
    rows = [{"file": name, "exists": (folder / name).exists(),
             "size (MB)": round((folder / name).stat().st_size / 1024 / 1024, 3)
                          if (folder / name).exists() else 0.0}
            for name in config.STEP11_FILES.values()]
    evaluation.show_table(pd.DataFrame(rows), note="Folder: %s" % folder)

    predictions = step11.get("predictions")
    if predictions is not None:
        evaluation.show_table(predictions.head(5), all_columns=True,
                              note="predictions.csv -- one row per test flow.")

---

### 11.3 Evaluation of Approach 2

**Purpose** — Score Approach 2 on the same untouched test set, with the same
evaluator.

**Input** — The saved Approach 2 model and the Step 8 test matrix.

**Method** — The **unchanged** Step 11 `evaluate()`: the same metrics, the same
duplicate-free subset, the same binary view and the same timing.

**Output** — `C:\MSCAD2022\outputs\step11_attention` — `metrics.json`,
`per_class_report.csv`, both confusion matrices, `predictions.csv`, the
attention tables and `approach_selection.json`.

**Why** — Two models scored by two evaluators cannot be compared. Approach 1's
numbers are read from its saved `step11/metrics.json` and re-checked afterwards
to prove nothing moved.

In [ ]:
step11b = models_attention.run_step11b()

#### 11.3a What the attention looked at

**Purpose** — Report the weights, and say what they can and cannot mean.

**Method** — Mean $\alpha$ per position, overall and per predicted class, plus
the entropy of each flow's weights.

**Why** — The entropy is the honesty check. A uniform distribution over $T$
positions has entropy $\ln T$; if the mean entropy sits at that value the
attention is flat and is telling the reader nothing, however pretty the heatmap
looks. The normalised figure below is $H / \ln T$: close to 1 means flat, close
to 0 means one position carries the decision.

In [ ]:
overall = step11b.get("attention")
if overall is not None and len(overall):
    models_attention.show_table(overall, all_columns=True,
        note="Mean attention weight per CNN feature-map position across the "
             "test set. These are not time steps.")
    entropy = (step11b.get("metrics") or {}).get("attention", {}).get("entropy")
    if entropy:
        models_attention.show_table(models_attention.kv_table([
            ("mean entropy", "%.4f" % entropy["mean"]),
            ("uniform would be", "%.4f" % entropy["uniform_would_be"]),
            ("normalised (H / ln T)", "%.4f" % entropy["normalised_mean"]),
            ("flows", "%s" % f"{entropy['flows']:,}"),
        ]), note="Close to 1 means the attention is flat and carries no "
                 "information; close to 0 means one position dominates.")

per_class = step11b.get("attention_per_class")
if per_class is not None and len(per_class):
    models_attention.show_table(per_class, all_columns=True,
        note="Mean attention per position, split by the class the model "
             "predicted.")

picture = config.STEP11B_OUTPUT_DIR / config.STEP11B_FILES["chart_attention"]
if picture.exists() and models_attention.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(picture)))
elif not picture.exists():
    print("missing:", picture)

### 11.4 Approach 1 against Approach 2

**Purpose** — Put the two proposed approaches side by side, and pick the
primary one without touching the test set.

**Input** — Approach 1's saved `step11/metrics.json` and Approach 2's.

**Method** — Metric by metric (full and duplicate-free), per-class F1
difference, and the two confusion matrices.

**Output** — `approach1_vs_approach2.csv`, the per-class difference and
`approach_selection.json`.

**Why** — **The primary model is chosen on validation macro-F1 alone.** Both
approaches are reported on the test set, but choosing between them on that
score would be selection on the test set, which is the cheapest way to turn an
honest comparison into an inflated one.

If Approach 2 does not win, this section says so. A negative result here is a
result: it says the attention block did not buy anything on this data, which is
worth more to a reader than a quiet omission.

In [ ]:
comparison_frame = step11b.get("comparison")
if comparison_frame is not None and len(comparison_frame):
    models_attention.show_table(comparison_frame, all_columns=True,
        note="`difference (A2 - A1)` is positive where Approach 2 scores "
             "higher; `favours` accounts for direction (lower is better for "
             "the FPR).")

difference = step11b.get("per_class_diff")
if difference is not None and len(difference):
    models_attention.show_table(difference, all_columns=True,
        note="Per-class F1, Approach 2 minus Approach 1, ordered by support. "
             "The rare classes move most on a single flow.")

selection = step11b.get("selection")
if selection:
    models_attention.show_table(models_attention.kv_table([
        ("primary approach", selection["primary"]),
        ("chosen on", selection["chosen_on"]),
        ("Approach 1 validation macro-F1",
         "n/a" if selection["approach1_validation_macro_f1"] is None
         else "%.4f" % selection["approach1_validation_macro_f1"]),
        ("Approach 2 validation macro-F1",
         "%.4f" % selection["approach2_validation_macro_f1"]),
        ("reason", selection["reason"]),
        ("test set used", selection["test_set_used"]),
    ]), note=selection.get("note", ""))

for key in ("chart_comparison", "chart_attention"):
    picture = config.STEP11B_OUTPUT_DIR / config.STEP11B_FILES[key]
    if picture.exists() and models_attention.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(picture)))

report = step11b.get("report")
if report is not None and len(report):
    models_attention.show_table(report, all_columns=True,
        note="Approach 1's saved metrics are re-read after Approach 2 runs and "
             "must be identical to 1e-9.")

### What Step 12 picks up

Step 12 compares baselines against this model. Three things keep that comparison
honest:

1. **Call the same functions.** `models.build_cnn_lstm` / `models.train_model`
   and `evaluation.evaluate` are reusable on purpose, so a difference between two
   rows of the results table is a difference between architectures, not between
   measurement harnesses.
2. **Use the same split, the same weights and the same seed.** They all come
   from Steps 5 and 9 and are already on disk.
3. **Quote both scores for every model**, full and duplicate-free, and lead with
   macro-F1 and G-mean rather than accuracy. A baseline that wins on accuracy and
   loses on G-mean has not won.


---

## Step 12 - Model comparison

**Purpose** — Compare the proposed approaches with five baselines, each
removing or replacing exactly one component, so that a difference between two
rows is attributable to that component and not to anything else.

**Input** — The Step 5 to Step 11 outputs on disk. Every configuration's
result is **read from its saved `metrics.json`**; nothing in this section
trains or retrains a model.

**Method** — Seven configurations, scored by the same Step 11 `evaluate()`,
assembled into one table by `comparison.build_full_comparison()`.

**Why** — Step 12 grew in three places (the original four configurations, the
Baseline 4 addendum and the Baseline 5 addendum), each with its own table and
its own row order. One table in one order is what a reader can actually use,
and it is built from the same saved files the earlier tables were built from,
so no number moves.

**Output** — `C:\MSCAD2022\outputs\step12` — one sub-folder per configuration,
plus `comparison_table_full.csv`, `per_class_f1_table_full.csv`,
`zero_recall_diagnostic_full.csv`, `head_to_head_full.csv`, `summary_full.md`
and the `_full` charts. The earlier tables and charts keep their own names and
are not overwritten.

> **Nothing is retrained here.** The `OVERWRITE_*` flags stay `False`, each
> configuration is loaded from disk, and `check_rows_unchanged()` asserts every
> rebuilt row against the saved `metrics.json` it came from (1e-9) and against
> the figures already in the write-up (4 decimals, 5e-5).

**The two proposed approaches.** Approach 1 is the hybrid CNN-LSTM of Step 10.
Approach 2 adds an attention block over the CNN feature map and is trained in
section 10.2; until it is trained its row reads `not yet trained - run 10.2`
and takes no part in the best-value marking, so the row order below is already
final.

### 12.1 Configurations

**Purpose** — Say precisely what each row is before any number is quoted.

**Input** — `config.STEP12_DISPLAY_ORDER` and the configuration specs.

**Method** — One row per configuration, in the order every table, chart and
legend in Step 12 uses.

**Output** — The table below.

**Why** — An ablation table is only readable if the reader can see which single
thing each row changes. The last column says exactly that.

In [ ]:
# Quieten TensorFlow's C++ logger for this section only.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from src.mscad import comparison

comparison.show_table(comparison.configuration_table(), all_columns=True,
    note="The display order used by every Step 12 table, chart and legend. "
         "`what it isolates` is the single component that row removes or "
         "replaces.")

In [ ]:
# Quieten TensorFlow's C++ logger for this section only.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from src.mscad import experiments

experiments.show_table(experiments.kv_table([
    ("output folder", str(config.STEP12_OUTPUT_DIR)),
    ("configurations", len(config.EXPERIMENTS)),
    ("BASELINE3_FEATURES", config.BASELINE3_FEATURES),
    ("OVERWRITE_STEP12", config.OVERWRITE_STEP12),
    ("N_SEEDS", config.N_SEEDS),
    ("INPUT_MODE", config.INPUT_MODE),
    ("imbalance", "%s / %s" % (config.IMBALANCE_STRATEGY,
                               config.CLASS_WEIGHT_SCHEME)),
]), note="With `OVERWRITE_STEP12 = False` a configuration whose metrics.json is "
         "already on disk is skipped, so the comparison is resumable.")

experiments.show_table(pd.DataFrame([
    {"config": e["label"], "feature selection": "Step 6 top 30"
     if e["feature_source"] == "selected30" else "none",
     "AE": "yes" if e["use_ae"] else "no",
     "CNN": "yes" if e["model_type"] == "cnn_lstm" else "no",
     "LSTM": "yes"}
    for e in config.EXPERIMENTS]))

### 12.2 Approach 1, B1, B2 and B3

One line per configuration: `[RUN ]` while it trains, `[SKIP]` if it was already
finished, `[DONE]` with its macro-F1 and elapsed time. Training four models takes
a while; `OVERWRITE_STEP12 = False` makes it resumable.


In [ ]:
step12 = experiments.run_step12()

### 12.2a Consistency

If any row here is `[FAIL]`, the comparison is not like-for-like and the table
below means nothing. These are checked, not assumed.


In [ ]:
checks = step12.get("consistency")
if checks is not None:
    experiments.show_table(checks)
    print()
    print("ALL CONSISTENCY CHECKS PASSED:", step12.get("consistency_ok"))
    print("test files untouched           :", step12.get("test_untouched"))

### 12.2b Across seeds

Only populated when `N_SEEDS >= 3`. Per-seed macro-F1 and the mean difference
from the proposed model, **as differences only** - with a handful of seeds there
is no basis for a significance claim, and none is made.


In [ ]:
seeds = step12.get("seed_comparison")
if seeds is not None and len(seeds):
    experiments.show_table(seeds,
        note="Differences, not tests. Set N_SEEDS = 3 or 5 in config.py to "
             "populate this.")
else:
    print("N_SEEDS = %d, so there is one run per configuration and no spread to"
          % config.N_SEEDS)
    print("report. Set N_SEEDS = 3 or 5 and re-run with OVERWRITE_STEP12 = True.")

---

### 12.3 Baseline 4: AE -> PCA -> LSTM

**Purpose** — Add a fifth configuration to the comparison: does a second,
linear compression of the autoencoder's latent space help, hurt, or do nothing?

**Input** — The four finished Step 12 configurations, read from disk, plus the
Step 5 features and the Baseline 1 encoder.

**Method** — Baseline 1's encoder (all → 15), then PCA fitted on the
**training** latent features only, then the same LSTM-only classifier Baseline 2
uses.

**Why** — Baseline 2 is AE → LSTM and Baseline 4 is AE → PCA → LSTM, sharing
one encoder and every hyperparameter. The gap between those two rows is PCA's
and nothing else's, which is the only way to answer the question honestly.

**Output** — `C:\MSCAD2022\outputs\step12\baseline4_ae_pca_lstm\` (model,
encoder, `pca.joblib`, `pca_variance.csv`, the scree plot, history, metrics,
per-class report, confusion matrices, predictions) plus the five-configuration
tables and charts **written beside** Step 12's own files, never over them.

> **Nothing in Step 12 is re-run or rewritten.** Its four `metrics.json` files
> are read from disk and its `comparison_table.csv` is left exactly as it was.
> Steps 13 and 14 address those four configurations by name and are unaffected:
> `EXPERIMENTS` still has four entries, and the five-configuration list lives
> beside it as `EXPERIMENTS_WITH_PCA`.

**What PCA does here.** On the training latent features $Z_{\text{train}}$ it
finds the orthogonal directions of greatest variance and keeps enough of them to
retain `PCA_N_COMPONENTS` of the total:

$$Z \;\longmapsto\; (Z - \mu_{\text{train}})\, W_k^{\top},
\qquad W_k \in \mathbb{R}^{k \times 15}$$

with $\mu_{\text{train}}$ and $W_k$ estimated from the training rows **only**.
A float setting (default `0.95`) is a variance target and $k$ is whatever meets
it; an int is a fixed $k$, and asking for more components than the latent space
has raises rather than silently clipping.

**The leakage guard is measured, not asserted.** `fit_pca_on_train` takes no
test argument at all, and the check below re-fits on the training rows (the
result must be identical) and then fits again *including* a corrupted copy of
the test rows (the result must differ). The second half matters: without it, the
first could pass on a PCA that is simply insensitive to its input.

In [ ]:
# Quieten TensorFlow's C++ logger for this section only.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from src.mscad import experiments_pca

experiments_pca.show_table(experiments_pca.kv_table([
    ("output folder", str(config.STEP12_OUTPUT_DIR / config.BASELINE4["name"])),
    ("configurations compared", len(config.EXPERIMENTS_WITH_PCA)),
    ("BASELINE4_AE_FEATURES", "%s (encoder from %s)"
                              % (config.BASELINE4_AE_FEATURES,
                                 config.BASELINE4["reuse_encoder_from"])),
    ("PCA_N_COMPONENTS", "%s (%s)"
                         % (config.PCA_N_COMPONENTS,
                            "fraction of variance kept"
                            if isinstance(config.PCA_N_COMPONENTS, float)
                            else "fixed number of components")),
    ("AE latent dim", config.AE_LATENT_DIM),
    ("classifier", "LSTM only, same hyperparameters as Baseline 2"),
    ("OVERWRITE_BASELINE4", config.OVERWRITE_BASELINE4),
]), note="`EXPERIMENTS` keeps its four Step 12 entries; Baseline 4 lives in "
         "`BASELINE4` and the five-configuration list is "
         "`EXPERIMENTS_WITH_PCA`.")

experiments_pca.show_table(experiments_pca.architecture_table(),
    all_columns=True,
    note="The five configurations. Baseline 2 and Baseline 4 differ in exactly "
         "one place, which is what makes the PCA comparison readable.")

### Run Baseline 4

Step 12's four configurations are **read from disk** — they are not retrained —
and only Baseline 4 is trained here. With `OVERWRITE_BASELINE4 = False` a
finished Baseline 4 is read back too, so this section is resumable.

If Step 12 has not been run at all, this stops and names the missing
`metrics.json` files rather than training a partial comparison.

In [ ]:
step12b = experiments_pca.run_step12_with_baseline4()

#### 12.3a Consistency

Baseline 4 has to be comparable with the four configurations it is placed
beside, so the same checks Step 12 ran are re-run with Baseline 4 included, plus
three of its own: the PCA leakage guard, the encoder provenance, and the
classifier input shape.

If any row here is `[FAIL]`, the comparison below is not like-for-like and the
table means nothing.

In [ ]:
checks = step12b.get("checks")
if checks is not None and len(checks):
    experiments_pca.show_table(checks, all_columns=True)
    print()
    print("ALL CHECKS PASSED      :", bool((checks["result"] == "[OK]").all()))
print("test files untouched   :", step12b.get("test_untouched", "(loaded)"))

#### 12.3b PCA on the autoencoder's latent space

**Purpose** — Report what the second compression actually kept.

**Input** — The training latent features.

**Method** — Explained variance per component and the running total.

**Output** — `pca_variance.csv` and `pca_scree.png` in the Baseline 4 folder.

**Why** — This is the number that decides how to read the Baseline 2 vs
Baseline 4 gap. If PCA kept 14 of 15 components, it did almost nothing and a
difference in score is noise; if it kept 5, it threw away two thirds of the
latent space and the score says whether that mattered.

The autoencoder has already compressed the features once, so there is no
guarantee the latent dimensions are redundant — that is the point of measuring
rather than assuming.

In [ ]:
variance = step12b.get("variance")
if variance is not None and len(variance):
    experiments_pca.show_table(variance, all_columns=True,
        note="Components are ordered by the variance they explain. The "
             "cumulative column is built from the rounded ratios beside it, so "
             "the column adds up to the total printed.")

picture = (config.STEP12_OUTPUT_DIR / config.BASELINE4["name"]
           / config.BASELINE4_FILES["pca_scree"])
if picture.exists() and experiments_pca.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(picture)))
elif not picture.exists():
    print("missing:", picture)

metrics = (step12b.get("baseline4") or {}).get("metrics") or {}
pca_info = metrics.get("pca") or {}
if pca_info:
    experiments_pca.show_table(experiments_pca.kv_table([
        ("setting", "%s (%s)" % (pca_info.get("requested"),
                                 pca_info.get("requested_kind"))),
        ("latent dimensions in", pca_info.get("latent_dim")),
        ("components kept", pca_info.get("n_components")),
        ("variance retained", "%.4f" % pca_info.get("variance_kept", float("nan"))),
        ("fitted on", pca_info.get("fitted_on")),
    ]), note="The PCA was fitted on the training rows only; validation and test "
             "features are only ever passed through `.transform()`.")

#### 12.3c What the PCA ablation can and cannot say

1. **It isolates PCA, and only PCA.** Baseline 2 and Baseline 4 share one
   encoder, one split, one set of class weights and every hyperparameter. The
   gap between them is attributable; a gap between Baseline 4 and the proposed
   configuration is not, because three things differ there at once.
2. **PCA is being applied to an already-compressed space.** The autoencoder has
   reduced the features once, non-linearly. A second, linear reduction may
   remove redundancy the autoencoder left behind, or may discard a direction the
   classifier was using. The components kept and the variance retained are
   reported above so the reader can see which is plausible.
3. **Variance is not usefulness.** PCA orders directions by variance, not by
   how well they separate the classes. A low-variance direction can carry the
   signal for a rare class — which is exactly the kind of class this dataset is
   short of, and exactly why the per-class table matters more here than the
   headline average.
4. **One seed per configuration.** The differences are single observations.
   `N_SEEDS = 3` or `5` is what would turn them into something quotable, and
   until then no significance is claimed.
5. **The duplicate-free columns remain the honest comparison**, for Baseline 4
   as for the other four.

---

### 12.4 Baseline 5: MI-only top 30 -> AE -> CNN-LSTM

**Purpose** — Isolate what the *fused* feature selection bought. The proposed
model ranks features with three signals — mutual information, a Pearson
redundancy filter and permutation importance. Baseline 5 keeps everything else
identical and selects with **MI alone**.

**Input** — Step 6's MI scores (read if saved, recomputed with Step 6's own
function if not), Step 6's proposed feature list, the Step 5 scaled matrices,
and the five finished configurations read from disk.

**Method** — MI top 30 → a new autoencoder (30 → 15) with Step 7's
hyperparameters → the Step 10 CNN-LSTM → the Step 11 evaluator.

**Why** — Proposed and Baseline 5 differ in exactly one place: how the 30
features were chosen. The gap between those two rows is what the Pearson filter
and the permutation importance bought, and nothing else. A gap between Baseline
5 and any other row is not attributable, because more than one thing differs
there.

**Output** — `C:\MSCAD2022\outputs\step12\baseline5\` (MI scores, the selected
30, the overlap table, encoder, model, history, metrics, per-class report,
confusion matrices, predictions) plus six-configuration tables and charts
written beside the Step 12 files with a `_6configs` suffix.

> **Nothing existing is re-run or rewritten.** The five finished configurations
> are read from `metrics.json`, their rows are asserted identical to Step 12's
> saved `comparison_table.csv`, and every Step 12 and Baseline 4 file is
> SHA-256 fingerprinted before and after to prove it.

> **If MI alone picks exactly the proposed 30, this stops.** Baseline 5 would
> then *be* the proposed model — same features, same autoencoder settings, same
> classifier, same seed — and reporting a difference of zero as a finding would
> be misleading. The honest result in that case is that on this dataset the
> Pearson filter and the permutation importance did not change the selection,
> and nothing is trained.

In [ ]:
# This addendum can run on its own after a kernel restart, so it sets up the
# import path itself rather than relying on an earlier cell.
import os
import sys
from pathlib import Path

_here = Path.cwd()
_project = _here if (_here / "config.py").exists() else _here.parent
if str(_project) not in sys.path:
    sys.path.insert(0, str(_project))

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

import pandas as pd
import config
from src.mscad import experiments_baseline5 as baseline5

baseline5.show_table(baseline5.kv_table([
    ("output folder", str(config.BASELINE5_OUTPUT_DIR)),
    ("configurations compared", len(config.EXPERIMENTS_WITH_PCA) + 1),
    ("selection", "mutual information only, top %d" % config.BASELINE5_TOP_K),
    ("MI settings", "n_neighbors = %d, random_state = %s (Step 6's)"
                    % (config.MI_N_NEIGHBORS, config.MI_RANDOM_STATE)),
    ("autoencoder", "%d -> %d, Step 7 hyperparameters"
                    % (config.BASELINE5_TOP_K, config.AE_LATENT_DIM)),
    ("classifier", "Step 10 CNN-LSTM, input (%d, 1)" % config.AE_LATENT_DIM),
    ("imbalance", "%s / %s (Step 9), no resampling"
                  % (config.IMBALANCE_STRATEGY, config.CLASS_WEIGHT_SCHEME)),
    ("OVERWRITE_BASELINE5", config.OVERWRITE_BASELINE5),
]), note="Baseline 5 is deliberately NOT in `EXPERIMENTS`: that list is what "
         "`experiments.run_step12()` iterates and what Steps 13 and 14 address "
         "by name.")

baseline5.show_table(baseline5.architecture_table(), all_columns=True,
    note="All six configurations. Proposed and Baseline 5 differ in exactly "
         "one column, which is what makes the comparison readable.")

### Run Baseline 5

The five finished configurations are **read from disk** — none is retrained —
and only Baseline 5 is trained here: the MI ranking, a new autoencoder and the
CNN-LSTM. With `OVERWRITE_BASELINE5 = False` a finished Baseline 5 is read back
too, so this section is resumable.

If Step 12 or the Baseline 4 addendum has not been run, this stops and names
the missing `metrics.json` files rather than training a partial comparison.

In [ ]:
step12c = baseline5.run_baseline5_comparison()

#### 12.4a The MI-only selection, and how far it differs

**Purpose** — Show which 30 features MI alone chose, and how much they overlap
with the proposed 30.

**Input** — Step 6's MI scores and its saved feature list.

**Method** — Rank by MI, take the top 30, compare the two sets.

**Output** — `selected_features_mi30.csv` and `feature_overlap.csv`.

**Why** — The overlap is the ceiling on how different the two configurations
can be. Two selections that share 29 of 30 features produce two models that are
nearly the same model, whatever the scores say — so this table is what tells you
how much weight the comparison below can carry.

In [ ]:
selected = step12c.get("selected")
if selected is not None and len(selected):
    baseline5.show_table(selected, all_columns=True,
        note="The MI-only top %d, ranked. Ties are broken alphabetically, so "
             "the selection is reproducible." % config.BASELINE5_TOP_K)

overlap = step12c.get("overlap")
if overlap is not None and len(overlap):
    counts = overlap["where"].value_counts().rename_axis("where")
    baseline5.show_table(counts.reset_index(name="features"),
        note="How the two selections divide up.")
    baseline5.show_table(overlap[overlap["where"] != "both"], all_columns=True,
        note="The features the two selections disagree about. Everything else "
             "is shared, so these are the whole of the difference between the "
             "proposed model and Baseline 5.")

metrics = (step12c.get("baseline5") or {}).get("metrics") or {}
selection = metrics.get("selection") or {}
if selection:
    baseline5.show_table(baseline5.kv_table([
        ("MI source", selection.get("mi_source")),
        ("shared with the proposed 30", selection.get("shared_with_proposed")),
        ("Jaccard index", "%.3f"
                          % selection.get("jaccard_with_proposed", float("nan"))),
        ("only in MI", ", ".join(selection.get("only_in_mi", [])) or "none"),
        ("only in proposed", ", ".join(selection.get("only_in_proposed", []))
                             or "none"),
    ]), note="A Jaccard index of 1.00 would have stopped the run: Baseline 5 "
             "would be the proposed model under another name.")

#### 12.4b Consistency

Baseline 5 has to be comparable with the five configurations it is placed
beside, so the same checks Step 12 ran are re-run with it included, plus four of
its own: the MI ranking came from training rows only, the selection really does
differ, the autoencoder is 30 → 15, and the five existing rows are **identical**
to Step 12's saved `comparison_table.csv`.

If any row here is `[FAIL]`, the comparison below is not like-for-like.

In [ ]:
checks = step12c.get("checks")
if checks is not None and len(checks):
    baseline5.show_table(checks, all_columns=True)
    print()
    print("ALL CHECKS PASSED      :",
          bool(checks["result"].isin(["[OK]", "[SKIP]"]).all()))
print("test files untouched   :", step12c.get("test_untouched", "(loaded)"))

report = step12c.get("report")
if report is not None and len(report):
    baseline5.show_table(report, all_columns=True,
        note="Read back from disk after writing. `the existing Step 12 files "
             "are byte-identical` is the add-only guarantee, checked by "
             "SHA-256 rather than asserted.")

#### 12.4c What the selection ablation can and cannot say

1. **It isolates the selection method, and only that.** Proposed and Baseline 5
   share the split, the scaler, the autoencoder hyperparameters, the classifier,
   the class weights, the seed and the evaluator. The gap between them is
   attributable; a gap between Baseline 5 and any other row is not.
2. **The overlap caps the result.** The two selections share most of their
   features, so the two models are largely the same model. A small difference is
   what this design can produce; a large one would be the surprise, and would be
   worth checking before believing.
3. **MI alone is not a straw man, and this is not a verdict on MI.** It is one
   signal measured per feature against the label. What it cannot see is
   redundancy between features — two near-identical columns both score highly
   and both get selected. That is precisely what the Pearson filter removes, so
   if the two selections differ mostly by redundant pairs, that is the
   mechanism, visible in the overlap table above.
4. **One seed, one split.** Every difference is a single observation.
   `N_SEEDS = 3` or `5` is what would turn it into something quotable.
5. **The rare classes decide this comparison and have about five test rows
   each.** A per-class difference of ±0.2 on five rows is one flow. Read the
   per-class table with the support column, not without it.
6. **The duplicate-free columns remain the honest comparison**, here as
   everywhere else in Step 12.

### 12.5 Full comparison table

**Purpose** — Every configuration, every metric, one table.

**Input** — The saved `metrics.json` of each configuration.

**Method** — `comparison.build_full_comparison()` reads them in the display
order; `check_rows_unchanged()` proves nothing moved.

**Output** — `comparison_table_full.csv` and `marked_comparison_table_full.csv`.

**Why** — Three tables in three orders cannot be compared by eye. This is the
table the write-up quotes.

`*` marks the best value in a column, on **every** tied row — a tie is a result
and naming one arbitrary winner would be a false statement. Lower is better for
the binary FPR, the parameter count and the two times.

In [ ]:
step12_full = comparison.run_step12_full()

### 12.6 Per-class F1

**Purpose** — Show where the macro average comes from.

**Input** — Each configuration's `predictions.csv`.

**Method** — Per-class F1 rebuilt from the predictions, so the full and
duplicate-free views are computed the same way for every row.

**Output** — `per_class_f1_table_full.csv` and the duplicate-free version.

**Why** — Macro-F1 gives a five-row class the same weight as a sixty-thousand
-row one. The per-class table is where a headline difference turns out to be
one rare class, or not.

In [ ]:
for frame, scope in ((step12_full.get("per_class"), "full test set"),
                     (step12_full.get("per_class_free"),
                      "duplicate-free subset")):
    if frame is not None and len(frame):
        comparison.show_table(frame, all_columns=True,
            note="Per-class F1 on the %s, in the display order. `test rows` is "
                 "the support." % scope)

### 12.7 Charts

**Purpose** — The same comparison, read at a glance.

**Input** — The full comparison table and the per-class table.

**Method** — Grouped bars (full and duplicate-free), the per-class heatmap,
training against inference time, and the row-normalised confusion matrices side
by side — all in the display order.

**Output** — the `_full` PNGs in `step12\`.

**Why** — The earlier charts covered four configurations and are kept under
their own names; these cover all of them and are the ones to put in the
write-up.

In [ ]:
for key in ("chart_metrics", "chart_metrics_free", "chart_per_class",
            "chart_time", "chart_confusion"):
    picture = config.STEP12_OUTPUT_DIR / config.STEP12_FULL_FILES[key]
    if picture.exists() and comparison.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(picture)))
    elif not picture.exists():
        print("missing:", picture)

### 12.8 Head to head

**Purpose** — The three comparisons that are actually attributable.

**Input** — The rows of the full table.

**Method** — Left minus right, metric by metric, with the direction accounted
for (lower is better for the FPR, the parameters and the times).

**Output** — `head_to_head_full.csv`.

**Why** — Each pair differs in exactly one thing:

| pair | what differs |
|---|---|
| Approach 1 vs B5 | the selection **method**: fused MI + Pearson + PI, against MI alone |
| Approach 1 vs B1 | the selection **step** itself: with, against without |
| Approach 1 vs Approach 2 | the architecture: CNN-LSTM, against CNN-Attention-LSTM |

A gap between any other pair is not attributable, because more than one thing
differs.

In [ ]:
head = step12_full.get("head_to_head")
if head is not None and len(head):
    for pair in head["pair"].unique():
        comparison.show_table(head[head["pair"] == pair], all_columns=True,
            note="%s. `favours` accounts for direction." % pair)
else:
    print("No head-to-head is available yet: it needs both rows of a pair.")

### 12.9 Discussion and summary

**Purpose** — State what the table supports, and nothing more.

**Input** — The computed tables above.

**Method** — `comparison.build_summary()` reads the numbers off them; the
zero-recall diagnostic names the classes behind every `G-mean = 0`.

**Output** — `summary_full.md` and `zero_recall_diagnostic_full.csv`.

**Why** — A discussion written by hand drifts from the table beside it the
moment anything is re-run.

**On `G-mean = 0`.** The G-mean is the geometric mean of the per-class recalls,
so it is zero the moment **one** class is never detected. A zero there does not
mean "poor overall" — it means a class was missed entirely, and the diagnostic
below names it.

In [ ]:
zero = step12_full.get("zero_recall")
if zero is not None and len(zero):
    comparison.show_table(zero, all_columns=True,
        note="Every configuration with G-mean = 0, and the class or classes it "
             "never detected. Reported for the full test set and the "
             "duplicate-free subset.")

summary = step12_full.get("summary")
if summary:
    if comparison.dl.in_notebook():
        from IPython.display import Markdown, display
        display(Markdown(summary))
    else:
        print(summary)

# the environment the comparison ran in, written by Step 12
environment_path = config.STEP12_OUTPUT_DIR / config.STEP12_FILES["environment"]
if environment_path.exists():
    import json
    environment = json.loads(environment_path.read_text(encoding="utf-8"))
    comparison.show_table(comparison.kv_table([
        ("python", environment["python"]),
        ("tensorflow / keras", "%s / %s" % (environment["tensorflow"],
                                            environment["keras"])),
        ("numpy / pandas / scikit-learn", "%s / %s / %s"
         % (environment["numpy"], environment["pandas"],
            environment["scikit-learn"])),
        ("devices", ", ".join(environment["devices"])),
        ("GPU available", environment["gpu_available"]),
    ]), note="Saved as environment.json by Step 12.")

### What Step 13 adds

Step 12 asked *which single model is best*. Step 13 asks a different question:
**can three imperfect models be combined so that each one is listened to where it
has earned the right to be?**

A simple average gives every model the same say on every flow. That is a
reasonable default and it is one of the baselines below, but it ignores two
things the pipeline already knows:

1. **Measured track record.** The validation set says how well each model handles
   each kind of traffic. One model may be strong on port scans and weak on web
   crawling, and the reverse may hold for another.
2. **Per-flow certainty.** A model that spreads its probability evenly across six
   classes is telling us it does not know; a model that puts 0.98 on one class is
   telling us it does.

Step 13 turns both into a weight that is recomputed **for every flow**, fuses the
three distributions with it, and reduces the result to one number per flow -- the
attack risk $R_t$ -- which is what an operator would actually watch.

---

## Step 13 - Adaptive Evidence Fusion

*Reliability-Aware Adaptive Evidence Fusion using Dynamic Model Weighting*

**Purpose** — Combine three heterogeneous models into one fused risk score
$R_t$ per flow, weighting each model per flow by its measured reliability in the
current behavioural context and its confidence on that flow.

**Input** — The models saved by Steps 10 and 12 (`proposed`,
`baseline2_ae_lstm`, `baseline3_cnn_lstm`), their feature pipelines, the Step 9
fit/validation split and the untouched Step 5 test set. **Nothing is retrained.**

**Method** — Per-context reliabilities and the decision threshold are measured
on the validation part only; the test set is scored once with those values held
fixed.

**Why** — A fixed-weight ensemble is as good as its average member. Weighting
by measured reliability and per-flow certainty lets a model that is strong in one
context carry that context, without a committee vote diluting it.

**Output** — `C:\MSCAD2022\outputs\step13` — `reliability_table.csv`,
`weights_summary.csv`, `fused_predictions.csv`, `comparison_table.csv`,
`ablation_table.csv`, `per_class_f1.csv`, `threshold.json`, `metrics.json`,
`step13_report.csv` and five charts.

> **Step 13 adds files; it changes nothing.** Steps 1-12 are finished. This
> section calls their functions, reads their outputs and writes only to
> `C:\MSCAD2022\outputs\step13\`. No model is retrained and no existing file is
> rewritten.

#### Which models the fusion uses, and why

`FUSION_MODELS` is `["proposed", "baseline2_ae_lstm", "baseline3_cnn_lstm"]`:
in the Step 12 display names, **Approach 1**, **B2** and **B3**. Those internal
keys are unchanged and this section is unchanged with them — Step 12's
reorganisation renamed nothing it depends on.

The three are chosen for **architectural diversity**, not for being the top
three on the Step 12 table. Approach 1 has the full CNN-LSTM over autoencoder
features, B2 drops the convolutions, and B3 drops the autoencoder: three models
that fail differently, which is the only thing that makes a weighted
combination worth more than its best member. Three models that agree everywhere
would fuse to no gain.

**B4 and B5 are comparison baselines, not fusion members.** B4 (AE -> PCA ->
LSTM) and B5 (MI-only selection) exist to attribute components in Step 12. They
are not fused, and adding them is not a change this section would make quietly:
`FUSION_MODELS` is a config setting, and changing it means re-running Steps 13
and 14 and republishing every number in them.

**Approach 2 is not in the fusion either.** It is trained in 10.2 and reported
in Steps 10 to 12. `step11_attention\\approach_selection.json` records which
approach wins on **validation** macro-F1; if that is Approach 2, it could be
swapped in by changing `FUSION_MODELS` in `config.py` — but it has not been,
deliberately. Steps 13 and 14 are published with Approach 1, and changing the
members means re-running both sections and republishing every number in them.
Nothing in Steps 13 and 14 is recomputed for Approach 2.

### 13.1 Purpose and method

Let $m = 1 \dots M$ index the models ($M = 3$), $c$ the classes ($C$ in total),
and let $B$ be the set of benign classes (those with `Label_binary` $= 0$).
Each model gives a softmax distribution $p_m(c \mid x)$ over classes for a flow
$x$.

**1. Each model's own attack risk**

$$r_m(x) \;=\; 1 - \sum_{c \in B} p_m(c \mid x)$$

Subtracting the benign mass rather than summing the attack classes keeps the
value exact when the row sums to 1, and cannot silently omit a class.

**2. The behavioural context $k(x)$**

$$k(x) \;=\; \arg\max_c \; \frac{1}{M}\sum_{m=1}^{M} p_m(c \mid x)
\qquad (\texttt{CONTEXT\_MODE = "consensus\_class"})$$

This is the models' own consensus, so it needs **no label** and is available at
test time. With `CONTEXT_MODE = "protocol"` the context is instead the flow's
protocol (TCP / UDP / ICMP / OTHER), read from the Step 5 metadata.

**3. Context reliability, measured on the validation part only**

$$\mathrm{Rel}_m(k) \;=\;
\frac{n_k \, F1_m(k) \;+\; K \, F1_m^{\text{macro}}}{n_k + K}$$

$F1_m(k)$ is model $m$'s F1 for class $k$ (consensus mode) or its macro-F1 over
the validation rows of protocol $k$ (protocol mode); $n_k$ is the number of
validation rows in that context; $K = $ `SHRINKAGE_K`; $F1_m^{\text{macro}}$ is
the model's macro-F1 over the whole validation set. When $n_k = 0$,
$\mathrm{Rel}_m(k) = F1_m^{\text{macro}}$.

*Why shrink.* A context seen twice yields an F1 of exactly 0 or 1 and nothing in
between. $K$ is the number of rows at which the context estimate and the global
figure count equally, so a rare context cannot swing a weight on the strength of
two examples.

**4. Per-flow confidence**

$$H\big(p_m(x)\big) = -\sum_c p_m(c \mid x) \ln p_m(c \mid x),
\qquad 0 \ln 0 := 0$$

$$\mathrm{Conf}_m(x) \;=\; 1 - \frac{H\big(p_m(x)\big)}{\ln C}$$

This is $1$ for a one-hot distribution and $0$ for a uniform one, so it lives on
the same $[0, 1]$ scale as the reliability it multiplies. With
`CONFIDENCE_TERM = False`, $\mathrm{Conf}_m(x) = 1$.

**5. The dynamic weight**

$$s_m(x) \;=\; \mathrm{Rel}_m\big(k(x)\big) \cdot \mathrm{Conf}_m(x)$$

$$w_m(x) \;=\; \frac{s_m(x)}{\sum_{j=1}^{M} s_j(x)}
\qquad (\texttt{WEIGHT\_MODE = "proportional"})$$

$$w_m(x) \;=\; \frac{\exp\big(s_m(x)/T\big)}{\sum_{j=1}^{M}\exp\big(s_j(x)/T\big)}
\qquad (\texttt{"softmax"}, \; T = \texttt{WEIGHT\_TEMPERATURE})$$

If $\sum_j s_j(x) < \varepsilon$ (`EPSILON`) the flow carries no usable evidence
either way and the weights fall back to $1/M$. The weights are non-negative and
sum to $1$ for every flow.

**6. The fused distribution and the risk**

$$P(c \mid x) \;=\; \sum_{m=1}^{M} w_m(x) \, p_m(c \mid x),
\qquad \hat{y}(x) = \arg\max_c P(c \mid x)$$

$$R_t \;=\; 1 - \sum_{c \in B} P(c \mid x)$$

**7. Optional temporal smoothing** (`TEMPORAL_SMOOTHING = True`): flows are
ordered by `Timestamp` within each `Source_Capture` and an EMA with
$\alpha = $ `EMA_ALPHA` is applied to $R_t$ **inside each capture only**. Because
the test set is a random subset, consecutive test flows have gaps -- which is why
this is off by default.

**8. The decision threshold** $\tau$ is chosen on the **validation** risk scores
(`THRESHOLD_RULE = "max_f1"`, or `"fpr_at_most"` with `TARGET_FPR`) and then
applied unchanged to the test set: a flow is flagged when $R_t \ge \tau$.

---

#### A worked example — *illustrative numbers, not results*

> The numbers in this box are **made up** to show the arithmetic. They are not
> produced by the cells below and say nothing about the MSCAD data.

Three models, three classes, class $0$ benign, `WEIGHT_MODE = "proportional"`:

| | $p_m(0)$ | $p_m(1)$ | $p_m(2)$ | $\mathrm{Rel}_m(k)$ |
|---|---|---|---|---|
| $M_1$ | 0.10 | 0.85 | 0.05 | 0.98 |
| $M_2$ | 0.40 | 0.50 | 0.10 | 0.90 |
| $M_3$ | 0.20 | 0.30 | 0.50 | 0.80 |

$\mathrm{Conf} \approx [0.528,\; 0.141,\; 0.063]$ — $M_1$ is the most decisive
and $M_3$ is nearly undecided.

$s = \mathrm{Rel} \cdot \mathrm{Conf} \approx [0.518,\; 0.127,\; 0.050]$, so
$w \approx [0.745,\; 0.183,\; 0.072]$.

$P \approx [0.162,\; 0.746,\; 0.092]$, the fused class is $1$, and
$R_t = 1 - 0.162 \approx 0.838$.

A simple average would have given $P(0) = 0.233$ and $R_t = 0.767$: the confident,
reliable model is what sharpens the decision. This example is asserted in
`tests/test_fusion.py` to a tolerance of $10^{-3}$, so the published arithmetic
and the code cannot drift apart.

---

> **One honest caveat, stated once and repeated in the summary.** The validation
> part used here was also used for early stopping in Step 10, so the reliability
> estimates are slightly **optimistic**. A fully clean design would hold out a
> third split for fusion alone; with a 23-row class in this dataset that would
> leave too few rows per class to estimate a per-context F1 at all. The honest
> move is to say so rather than to hide it.

In [ ]:
# Quieten TensorFlow's C++ logger for this section only.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from src.mscad import fusion

fusion.show_table(fusion.kv_table([
    ("output folder", str(config.STEP13_OUTPUT_DIR)),
    ("models fused", ", ".join(config.FUSION_MODELS)),
    ("RELIABILITY_METRIC", config.RELIABILITY_METRIC),
    ("SHRINKAGE_K", config.SHRINKAGE_K),
    ("CONFIDENCE_TERM", config.CONFIDENCE_TERM),
    ("WEIGHT_MODE", "%s (T = %s)" % (config.WEIGHT_MODE,
                                     config.WEIGHT_TEMPERATURE)),
    ("CONTEXT_MODE", config.CONTEXT_MODE),
    ("THRESHOLD_RULE", "%s (TARGET_FPR = %s)" % (config.THRESHOLD_RULE,
                                                 config.TARGET_FPR)),
    ("TEMPORAL_SMOOTHING", "%s (alpha = %s)" % (config.TEMPORAL_SMOOTHING,
                                                config.EMA_ALPHA)),
    ("EPSILON", config.EPSILON),
    ("OVERWRITE_STEP13", config.OVERWRITE_STEP13),
]), note="Every setting above comes from the Step 13 block in `config.py`. "
         "Nothing here is hard-coded in the module.")

### Run the fusion

This loads the three saved models, predicts on the validation part and on the
test set, measures the reliabilities, picks $\tau$, fuses, compares and runs the
ablations. It trains nothing, so it is quick compared with Steps 10 and 12.

With `OVERWRITE_STEP13 = False` a completed run is read back from disk instead of
being recomputed, which makes this section resumable.

In [ ]:
step13 = fusion.run_step13()

### 13.2 Validation predictions and context reliability

**Purpose** — Measure how well each model handles each context, using
validation rows only.

**Input** — Each model's probabilities on the Step 9 validation rows. The split
is a pure function of `(y_train, CLF_VAL_SIZE, RANDOM_STATE)`, so asking Step 9
for it again selects exactly the rows Steps 10-12 validated on — which is what
makes the three reliabilities comparable.

**Method** — $F1_m(k)$ per context, shrunk towards $F1_m^{\text{macro}}$ with
$K = $ `SHRINKAGE_K`.

**Output** — `reliability_table.csv` and the reliability heatmap.

**Why** — This is the only place a *track record* enters the fusion, and it must
come from data the models were not scored on. The alignment table below is the
guard: fusing frames that disagree on row order produces numbers that look
plausible and mean nothing, so a mismatch raises rather than warns.

In [ ]:
reliability = step13.get("reliability")
if reliability is not None and len(reliability):
    fusion.show_table(reliability, all_columns=True,
        note="`n_validation_rows` is what the shrinkage responds to: where it is "
             "small, `reliability` sits close to `macro_f1_global` rather than to "
             "the raw per-context figure.")

match = step13.get("probability_match")
if match is not None and len(match):
    fusion.show_table(match,
        note="Recomputed test probabilities against the Step 12 predictions.csv. "
             "A `[FAIL]` would mean the saved model is not the one that produced "
             "those predictions.")

picture = config.STEP13_OUTPUT_DIR / config.STEP13_FILES["chart_reliability"]
if picture.exists() and fusion.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(picture)))
elif not picture.exists():
    print("missing:", picture)

### 13.3 Per-flow confidence and dynamic weights

**Purpose** — Turn reliability and per-flow certainty into one weight per model
per flow.

**Input** — The reliabilities from 13.2 and each model's test-set distribution.

**Method** — $s_m = \mathrm{Rel}_m(k) \cdot \mathrm{Conf}_m$, normalised across
models; equal weights where $\sum_j s_j < \varepsilon$.

**Output** — `weights_summary.csv` and the mean-weight chart.

**Why** — The **spread** is the whole point. If the mean weights are flat and
their standard deviations are near zero, this is a fixed-weight ensemble wearing
a different name, and the comparison below will show it.

In [ ]:
summary = step13.get("weights_summary")
if summary is not None and len(summary):
    by_context = summary[summary["grouping"] == "context"]
    by_class = summary[summary["grouping"] == "true class"]
    fusion.show_table(by_context, all_columns=True,
        note="Mean +- std of the weight each model received, grouped by the "
             "context the fusion assigned. Every group's mean weights sum to 1.")
    fusion.show_table(by_class, all_columns=True,
        note="The same weights grouped by the TRUE class instead. This is the "
             "diagnostic: it says whether the model that was listened to in a "
             "context is the one that was actually right there.")

picture = config.STEP13_OUTPUT_DIR / config.STEP13_FILES["chart_weights"]
if picture.exists() and fusion.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(picture)))
elif not picture.exists():
    print("missing:", picture)

### 13.4 Fused risk score $R_t$ and the decision threshold

**Purpose** — Reduce the fused distribution to one number per flow and decide
where to cut it.

**Input** — The fused $P(c \mid x)$ and the benign class codes.

**Method** — $R_t = 1 - \sum_{c \in B} P(c \mid x)$; $\tau$ chosen on the
**validation** risk scores by `THRESHOLD_RULE`, then applied unchanged to test.

**Output** — `fused_predictions.csv`, `threshold.json` and the risk
distribution chart.

**Why** — $R_t$ is the operational output: one score per flow that an analyst
can rank by. Choosing $\tau$ on validation is what keeps the reported FPR
honest — a threshold tuned on the test set would report a false-positive rate it
could never reproduce in service.

In [ ]:
threshold = step13.get("threshold") or {}
if threshold:
    fusion.show_table(fusion.kv_table([
        ("rule", threshold.get("rule")),
        ("tau", "%.6f" % threshold.get("tau", float("nan"))),
        ("candidates considered", threshold.get("candidates")),
        ("validation precision", "%.4f" % threshold.get("validation_precision",
                                                        float("nan"))),
        ("validation recall", "%.4f" % threshold.get("validation_recall",
                                                     float("nan"))),
        ("validation F1", "%.4f" % threshold.get("validation_f1", float("nan"))),
        ("validation FPR", "%.4f" % threshold.get("validation_fpr", float("nan"))),
        ("chosen on", threshold.get("chosen_on")),
    ]), note="Chosen on the validation part and applied unchanged to the test "
             "set. No test row took part in this choice.")

predictions = step13.get("fused_predictions")
if predictions is not None and len(predictions):
    fusion.show_table(predictions.head(8), all_columns=True,
        note="The first eight test flows: each model's own risk `r_m`, its "
             "reliability and confidence, the weight it received, the fused "
             "distribution, `R_t` and the decision. %s rows in all."
             % f"{len(predictions):,}")

picture = config.STEP13_OUTPUT_DIR / config.STEP13_FILES["chart_risk"]
if picture.exists() and fusion.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(picture)))
elif not picture.exists():
    print("missing:", picture)

### 13.5 Comparison with single models and fixed fusion

**Purpose** — Say whether the adaptive weighting earns its keep.

**Input** — The same untouched test set, scored with the same Step 11
`evaluate()` helpers for every configuration.

**Method** — Each single model; three fixed-weight fusions (simple average,
majority vote with ties broken by the highest mean probability, and static
weights from the validation macro-F1); and the adaptive fusion.

**Output** — `comparison_table.csv`, `per_class_f1.csv` and the comparison bar
charts, full and duplicate-free.

**Why** — Lead with macro-F1 and G-mean rather than accuracy, and quote the
duplicate-free columns as the conservative estimate. A fusion that wins on
accuracy and loses on G-mean has not won: it has learned to agree with the
majority class.

> The majority-vote row takes its *class* from the vote and its *risk* from the
> simple average, because a vote produces no distribution to take a risk from.
> Its binary columns are therefore the simple average's; only its multiclass
> columns are the vote's.

In [ ]:
comparison = step13.get("comparison")
if comparison is not None and len(comparison):
    fusion.show_table(comparison, all_columns=True,
        note="Single models, fixed-weight fusion and adaptive fusion on the same "
             "untouched test set. `fusion overhead (ms/1k)` excludes the models' "
             "own inference time.")

per_class = step13.get("per_class")
if per_class is not None and len(per_class):
    fusion.show_table(per_class, all_columns=True,
        note="Per-class F1 by configuration. `test rows` is the support; where it "
             "is ~5, a single flow changing hands moves the F1 a long way.")

for key in ("chart_comparison", "chart_comparison_free", "chart_curves"):
    picture = config.STEP13_OUTPUT_DIR / config.STEP13_FILES[key]
    if picture.exists() and fusion.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(picture)))
    elif not picture.exists():
        print("missing:", picture)

### 13.6 Ablation study

**Purpose** — Attribute whatever the fusion achieves to the parts it is made
of.

**Input** — The same test probabilities, re-fused with one setting changed at a
time.

**Method** — Without the confidence term; without the context (global
reliability everywhere); the other `CONTEXT_MODE`; and softmax weights at
$T = 0.2$ and $T = 1.0$.

**Output** — `ablation_table.csv`.

**Why** — Each row removes exactly one ingredient, so a gap points at that
ingredient. Two readings matter: if *no confidence term* scores the same as the
default, the entropy term is decoration; if *no context* scores the same, the
per-context reliability is decoration. Either would be worth knowing, and
neither is hidden here.

> The `context = protocol` row needs the `Protocol` column in the Step 5
> metadata. If it is absent the row is **skipped and said to be skipped**, never
> guessed.

In [ ]:
ablation = step13.get("ablation")
if ablation is not None and len(ablation):
    columns = ["ablation", "F1 (macro)", "G-mean", "ROC-AUC (macro OvR)",
               "binary F1", "binary FPR", "binary PR-AUC",
               "F1 macro (dup-free)"]
    fusion.show_table(ablation[[c for c in columns if c in ablation.columns]],
        all_columns=True,
        note="Row 1 is the default configuration; each later row changes exactly "
             "one setting. Compare each row against row 1, not against the "
             "single models.")

### 13.7 Summary

**Purpose** — State what the numbers above support, and nothing more.

**Input** — The comparison, ablation and weight tables just computed.

**Method** — The text below is composed by `fusion.step13_summary()` from those
tables: every figure in it is read off a computed table, and the direction of
every comparison (*above*, *level with*, *below*) is decided by the numbers
rather than written in advance.

**Output** — Rendered here and stored in `metrics.json`, so the resumable path
shows the same text.

**Why** — A summary written by hand drifts from the tables beside it as soon as
anything is re-run. One generated from the tables cannot.

In [ ]:
text = (step13.get("metrics") or {}).get("summary")
if text:
    if fusion.dl.in_notebook():
        from IPython.display import Markdown, display
        display(Markdown(text))
    else:
        print(text)
else:
    print("No summary stored. Re-run with OVERWRITE_STEP13 = True.")

report = step13.get("report")
if report is not None and len(report):
    fusion.show_table(report, all_columns=True,
        note="Read back from disk after writing: row counts, weights summing to "
             "1, fused rows summing to 1 and R_t within [0, 1]. A `[FAIL]` here "
             "means the saved file is wrong, whatever the in-memory numbers said.")

### What Step 13 can and cannot say

1. **The reliabilities are slightly optimistic.** They are measured on the
   validation part, which Step 10 also used for early stopping. Treat the
   reliability table as a well-founded ordering of the models per context, not as
   an unbiased estimate of their F1.
2. **It cannot beat a ceiling that is already reached.** Where the single models
   are near-perfect on the full test set, the fused score has nowhere to go. The
   duplicate-free subset and the per-class table are where a real difference
   shows up, which is why both are quoted.
3. **Fusion adds no new information.** It re-weights what three models already
   said. If all three fail on a class, no weighting recovers it — the per-class
   table shows that directly.
4. **The context is a consensus, not a ground truth.** `k(x)` is the models' own
   agreement, so a flow all three misread is weighted by the reliability of the
   wrong context. That is the cost of a label-free context, and it is why the
   protocol-based alternative is run as an ablation rather than dismissed.
5. **$R_t$ is one number, and one number loses detail.** It answers "how
   suspicious is this flow", not "what is it". The multiclass columns and the
   fused class in `fused_predictions.csv` are what answer the second question.
6. **One split, one seed.** Every figure here comes from a single train/test
   split and the models trained in Steps 10 and 12. The ablation differences are
   single observations, not significance tests, and none is claimed.

### What Step 14 adds

Step 13 ends with a number: $R(x) = 0.94$, raise the alert. An analyst reading
that number has no way to act on it. They need to know **why** -- which
properties of this flow made three models suspicious, which model carried the
decision, and what to look at first.

Step 14 turns each alert into an evidence report. It adds no model, changes no
threshold and overturns no Step 13 decision; it reads the saved models and the
saved fusion and explains what they already did.

Three things it deliberately does **not** do, because this pipeline cannot
support them:

- it does not claim a multi-stage attack progression (the attack stage is a
  lookup of the predicted label, nothing more);
- it does not reason over sequences, tokens or graphs (there is no such model
  here);
- it does not report an analyst triage time (nothing here was measured on an
  analyst).

---

## Step 14 - Explainability Layer

**Purpose** — Turn every ATTACK and SUSPICIOUS alert into a human-readable
evidence report: which input features drove each model, how the adaptive fusion
combined them into $R(x)$, how far the evidence can be trusted, and what an
analyst should check next.

**Input** — The Step 6 selected features and the Step 5 scaled and **unscaled**
test matrices, the Step 5 metadata, the Step 7 encoder, the Step 10 and Step 12
models and encoders, and the Step 13 fused predictions, reliabilities and
threshold. Everything is aligned on `Orig_Row`.

**Method** — Integrated Gradients implemented directly on `tf.GradientTape`,
on a frozen composed model that maps the **original named features** through the
encoder and the classifier, so the attributions are on `Flow Duration` rather
than on latent unit 7. Each attribution is then checked against direct
perturbation, which uses no gradients at all.

**Why** — An alert without a reason is not actionable, and an explanation
nobody has checked is worse than none. Every attribution here is verified three
ways: completeness ($\sum_i \mathrm{IG}_i = F(x) - F(x')$), agreement with
perturbation, and stability under small noise.

**Output** — `C:\MSCAD2022\outputs\step14` — `evidence_records.csv`,
`explanations.csv`, `global_feature_importance.csv`,
`explanation_validation.csv`, `model_agreement.csv`, `alert_explanations.csv`,
`example_alert_reports.md`, `thresholds_step14.json`,
`explanation_metrics.json`, `step14_report.csv` and seven charts.

> **Step 14 adds files; it changes nothing.** The models, the encoders, the
> Step 13 weights and $\tau$ are fixed. Nothing is retrained or fine-tuned, and
> no Step 13 decision changes.

#### Which model is explained here

Step 14 explains the Step 13 fusion, so it covers the same three models:
**Approach 1**, **B2** and **B3**. Approach 2 (CNN-Attention-LSTM) is reported
in Steps 10 to 12 and is **not** part of the fusion or of these explanations;
nothing in Steps 13 and 14 is recomputed for it.

### 14.1 Input and evidence collection

**Purpose** — Gather every saved output an explanation needs, prove they
describe the same flows, and decide which alerts to explain.

**Input** — Steps 5 to 13, aligned on `Orig_Row`.

**Method** — Decision bands, then a deterministic stratified selection.

**Output** — `evidence_records.csv`.

**Why** — Fusing or explaining frames that disagree on row order produces a
report that is fluent and wrong, so the alignment is asserted and raises rather
than warns.

**The decision bands.** Step 13's attack rule is kept exactly:

$$\text{band}(x) = \begin{cases}
\textsf{ATTACK} & R(x) \ge \tau \quad\text{(Step 13, unchanged)}\\[2pt]
\textsf{SUSPICIOUS} & \tau_s \le R(x) < \tau\\[2pt]
\textsf{BENIGN} & R(x) < \tau_s
\end{cases}$$

$\tau$ comes from Step 13 and is **not recomputed**. $\tau_s$ is new, and is
chosen on the **validation** part of the Step 9 split alone: the lowest risk at
which validation attack recall still reaches `SUSPICIOUS_RECALL_TARGET`. The
suspicious band only subdivides what Step 13 already called benign, so no Step
13 decision is altered -- it adds a second, lower shelf for an analyst who wants
to see what sat just under the line.

**Which flows get explained.** Every alert, up to `EXPLAIN_SAMPLE_PER_CLASS`
per fused class, drawn from a seeded generator so two runs explain the same
flows; a class with fewer alerts than that contributes all of them, which is how
the rare classes stay represented. A small benign sample is added for contrast.

In [ ]:
# Quieten TensorFlow's C++ logger for this section only.
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

from src.mscad import explainability

explainability.show_table(explainability.kv_table([
    ("output folder", str(config.STEP14_OUTPUT_DIR)),
    ("models explained", ", ".join(config.FUSION_MODELS)
                         if config.EXPLAIN_OTHER_MODELS
                         else config.FUSION_MODELS[0]),
    ("IG_STEPS (trapezoidal)", config.IG_STEPS),
    ("IG_BASELINE", config.IG_BASELINE),
    ("TOP_K_FEATURES", config.TOP_K_FEATURES),
    ("COMPLETENESS_TOL", config.COMPLETENESS_TOL),
    ("PERTURB_MODE", config.PERTURB_MODE),
    ("stability", "%s repeats at noise %s"
                  % (config.STABILITY_N_REPEATS, config.STABILITY_NOISE_STD)),
    ("EXPLAIN_SAMPLE_PER_CLASS", config.EXPLAIN_SAMPLE_PER_CLASS),
    ("SUSPICIOUS_BAND", "%s (recall target %s)"
                        % (config.SUSPICIOUS_BAND,
                           config.SUSPICIOUS_RECALL_TARGET)),
    ("confidence thresholds", "High >= %s, Medium >= %s"
                              % (config.CONF_HIGH_PROB, config.CONF_MEDIUM_PROB)),
    ("SOURCE_CONTEXT", "%s (%s minute window)"
                       % (config.SOURCE_CONTEXT, config.SOURCE_CONTEXT_WINDOW_MIN)),
    ("OVERWRITE_STEP14", config.OVERWRITE_STEP14),
]), note="Every setting comes from the Step 14 block in `config.py`. "
         "`tau` is NOT here: it is Step 13's and is read from "
         "`step13/threshold.json` unchanged.")

### Run the explainability layer

This loads the three saved models, rebuilds each as a composed differentiable
function on its **original** features, checks that each one still reproduces the
probabilities Step 12 saved, attributes, validates, explains the fusion and
writes the reports.

Integrated Gradients takes `IG_STEPS + 1` gradient evaluations per flow per
model per target, so this is the slowest part; it still trains nothing. With
`OVERWRITE_STEP14 = False` a finished run is read back from disk instead.

In [ ]:
step14 = explainability.run_step14()

### 14.2 Feature attribution with Integrated Gradients

**Purpose** — Say which input features moved each model on each alert.

**Input** — The composed model and a baseline flow $x'$.

**Method** — Integrated Gradients, implemented on `tf.GradientTape`.

**Output** — `explanations.csv`, `global_feature_importance.csv` and three
charts.

**Why** — A gradient at a single point is noisy and does not add up to
anything. IG integrates the gradient along the straight path from a reference
flow to this one, which gives it the completeness property below -- the one
thing that makes an attribution auditable.

$$\mathrm{IG}_i(x) \;=\; (x_i - x'_i)\int_0^1
\frac{\partial F\big(x' + \alpha (x - x')\big)}{\partial x_i}\, d\alpha$$

evaluated with the trapezoidal rule over `IG_STEPS` intervals. $F$ is one of
two scalars, both reported:

$$F_{\text{class}}(x) = p_m\big(\hat{c} \mid x\big)
\qquad\text{and}\qquad
F_{\text{risk}}(x) = r_m(x) = 1 - \sum_{c \in B} p_m(c \mid x)$$

**Completeness** is the identity that makes the attribution checkable:

$$\sum_i \mathrm{IG}_i(x) \;=\; F(x) - F(x')$$

Every flow's relative error $\big|\sum_i \mathrm{IG}_i - (F(x) -
F(x'))\big| \big/ |F(x) - F(x')|$ is measured and compared with
`COMPLETENESS_TOL`; failures are flagged, not hidden, and a flow whose
attributions do not add up cannot be marked **High** confidence.

**The baseline $x'$** is the per-feature median of the **training benign**
rows -- "a typical normal flow", which is the comparison an analyst makes
anyway. `zeros` and `train_mean` are available as ablations. The baseline is
built from training rows only, and its own risk $F(x')$ is reported below: it
should be low, because if the reference point already looks like an attack then
every attribution is measured from the wrong place.

**On the autoencoder.** The 15 latent units have no meaning an analyst can act
on, so the encoder is placed **inside** the differentiated function:

$$x\ (30\ \text{named features}) \;\to\; \text{encoder} \;\to\;
\text{reshape} \;\to\; \text{CNN-LSTM} \;\to\; \text{logits}
\;\to\; \text{softmax}$$

and the composed model is asserted to reproduce the saved Step 11/12
probabilities to $10^{-5}$ before anything is attributed. If it did not, the
attributions would explain a different model from the one that raised the alert.

**Temporal attribution** is computed only when `INPUT_MODE = "time_window"`.
In `feature_seq` mode a flow is one vector with no time steps, so there is
nothing to attribute over time and the chart is skipped rather than faked.

In [ ]:
importance = step14.get("global_importance")
if importance is not None and len(importance):
    overall = importance[
        (importance["scope"] == "all explained flows")
        & (importance["target"] == explainability.TARGET_RISK)]
    for model in overall["model"].unique():
        explainability.show_table(
            overall[overall["model"] == model]
            .nlargest(10, "mean_abs_ig")[["feature", "mean_abs_ig",
                                          "mean_signed_ig", "flows"]],
            note="`%s`: `mean_abs_ig` is how much the feature mattered, "
                 "`mean_signed_ig` which way it pushed the attack risk." % model)

for key in ("chart_global", "chart_heatmap", "chart_signed"):
    picture = config.STEP14_OUTPUT_DIR / config.STEP14_FILES[key]
    if picture.exists() and explainability.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(picture)))
    elif not picture.exists():
        print("missing:", picture)

### 14.3 Validating the explanations

**Purpose** — Check the attributions against a method that shares none of
their assumptions.

**Input** — The same flows, the same baseline.

**Method** — Replace one feature at a time with its baseline value and measure
the drop in $F$; then rank-compare those drops with $|\mathrm{IG}|$.

**Output** — `explanation_validation.csv`, the scatter and the deletion bars.

**Why** — IG is a gradient method, so it can be confidently wrong in ways a
gradient method cannot see. Perturbation uses no gradients at all: where the two
agree, the attribution is describing the model rather than the method.

Three numbers come out of this:

1. **Spearman** between $|\mathrm{IG}|$ and the perturbation drops, per flow.
   Flows with a negative correlation are counted and reported -- on those, the
   two methods disagree outright and the explanation should be read with care.
2. **Top-$K$ overlap**: how much of the top-5 the two rankings share.
3. **The deletion check**: setting the top-$K$ attributed features to the
   baseline should cost the model more than setting $K$ random features to it.
   If it does not, the ranking is not informative, and the summary says so in
   those words rather than quietly moving on.

In [ ]:
validation = step14.get("validation")
if validation is not None and len(validation):
    summary = (validation.groupby("model")
               .agg(mean_spearman=("spearman", "mean"),
                    median_spearman=("spearman", "median"),
                    mean_top_k_overlap=("top_k_overlap", "mean"),
                    flows_disagreeing=("spearman", lambda s: int((s < 0).sum())),
                    flows=("spearman", "size"))
               .reset_index())
    explainability.show_table(summary, all_columns=True,
        note="`flows_disagreeing` counts flows where IG and perturbation rank "
             "the features in opposite orders (Spearman < 0).")

metrics = step14.get("metrics") or {}
deletion = metrics.get("deletion_check") or {}
if deletion:
    explainability.show_table(pd.DataFrame([
        {"model": model, "K": value["k"],
         "mean drop, top-K by |IG|": value["mean_drop_top_k"],
         "mean drop, K random": value["mean_drop_random_k"],
         "top-K drops more": value["top_k_drops_more"]}
        for model, value in deletion.items()]), all_columns=True,
        note="Higher is better in the first column. `False` in the last column "
             "would mean the attribution is not beating chance for that model.")

for key in ("chart_scatter", "chart_deletion"):
    picture = config.STEP14_OUTPUT_DIR / config.STEP14_FILES[key]
    if picture.exists() and explainability.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(picture)))
    elif not picture.exists():
        print("missing:", picture)

### 14.4 Explaining the adaptive fusion, exactly

**Purpose** — Show how three models' opinions became one risk score.

**Input** — The weights, reliabilities and confidences Step 13 saved.

**Method** — Split $R(x)$ into one contribution per model.

**Output** — `model_agreement.csv` and the stacked contribution chart.

**Why** — This is the one part of the explanation that is exact rather than
approximate, and it is worth saying why. Each model's contribution is

$$C_m(x) \;=\; w_m(x)\, r_m(x)$$

and because the weights sum to one,

$$\sum_m C_m(x) \;=\; \sum_m w_m (1 - \textstyle\sum_{c \in B} p_m(c))
\;=\; 1 - \sum_{c \in B} \sum_m w_m\, p_m(c)
\;=\; 1 - \sum_{c \in B} P(c) \;=\; R(x)$$

So the contributions are not an approximation of the risk: they **are** the
risk, split by model. The code checks this against Step 13's own
`fused_predictions.csv` to $10^{-6}$ and raises on a mismatch -- which also
makes it a check on Step 13's saved file rather than a restatement of Step 14's
arithmetic.

The weights, reliabilities and confidences are **read back from Step 13**, not
recomputed, so the report explains the numbers that actually produced the alert.
(The reliabilities are recomputed once as a cross-check and must match.)

In [ ]:
agreement = step14.get("agreement")
if agreement is not None and len(agreement):
    explainability.show_table(agreement.head(12), all_columns=True,
        note="The first twelve explained flows. `dominant_model` is the largest "
             "contribution C_m = w_m r_m, which is not always the most "
             "confident model.")
    counts = agreement["agreement"].value_counts().rename_axis("agreement")
    explainability.show_table(counts.reset_index(name="flows"),
        note="How often the three models agreed on the class, across every "
             "explained flow.")
    if "fusion_changed_the_class" in agreement.columns:
        changed = int(agreement["fusion_changed_the_class"].sum())
        print("Adaptive weighting changed the predicted class away from the "
              "simple average on %d of %d explained flows."
              % (changed, len(agreement)))

picture = config.STEP14_OUTPUT_DIR / config.STEP14_FILES["chart_contributions"]
if picture.exists() and explainability.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(picture)))
elif not picture.exists():
    print("missing:", picture)

### 14.5 The per-alert evidence report

**Purpose** — Put everything above in front of an analyst, one alert at a time.

**Input** — The attributions, the fusion breakdown and the Step 5 metadata.

**Method** — A fixed set of evidence rows, each with a confidence label.

**Output** — `alert_explanations.csv` and `example_alert_reports.md`.

**Why** — A report with a different shape for every alert cannot be scanned.
These rows are the same every time, and the ones this pipeline cannot support
are absent by construction, not by accident.

Every report carries exactly these rows: **Alert ID**, **Risk score**, one row
per model, **Fusion**, **Attack family**, **Attack stage**, **Flow context**,
**Source context**, **Uncertainty** and **Recommended action**.

**The confidence column** is a rule, not a judgement:

$$\textsf{High} \iff \max_c P(c \mid x) \ge \texttt{CONF\_HIGH\_PROB}
\;\wedge\; \text{all models agree}
\;\wedge\; \text{completeness passed}$$

$$\textsf{Medium} \iff \max_c P(c \mid x) \ge \texttt{CONF\_MEDIUM\_PROB}
\;\vee\; \text{a majority agrees}
\qquad\qquad \textsf{Low} \iff \text{otherwise}$$

Three things in the report are **context, not evidence**, and each says so on
its own line:

- **Flow context** prints Src/Dst IP, ports, protocol and timestamp. No model
  ever sees them. The MSCAD labels were assigned by attacker IP, so a model
  given Src IP would simply read the answer off it.
- **Attack stage** is `ATTACK_STAGE_MAP[class]`: a label-to-tactic lookup. An
  unmapped label reports `Unmapped`; nothing is guessed, and no progression is
  implied.
- **Source context** lists other alerts from the same Src IP in the test subset.
  The test set is a random 20% of the flows, so a gap means "not in this
  subset", never "the source went quiet".

**Recommended action** is a suggested check for a human, prefixed *"Suggested
analyst check:"*. Nothing in this project executes it.

In [ ]:
alerts = step14.get("alerts")
if alerts is not None and len(alerts):
    detail_columns = [c for c in alerts.columns if c.startswith("row: ")]
    shown = set()
    for _i, alert in alerts.iterrows():
        family = alert["fused_label"]
        if family in shown:
            continue
        shown.add(family)
        table = pd.DataFrame({
            "Evidence component": [c.replace("row: ", "") for c in detail_columns],
            "Detail": [alert[c] for c in detail_columns]})
        explainability.show_table(table, all_columns=True,
            note="Alert %s -- %s, band %s, confidence %s."
                 % (alert["alert_id"], family, alert["band"],
                    alert["confidence"]))
    print()
    print("Plain-text rendering of the same reports (also saved as "
          "example_alert_reports.md):")
    print()
    text = step14.get("examples_markdown", "")
    for block in text.split("```")[1::2]:
        print(block.strip())
        print()
else:
    print("No ATTACK or SUSPICIOUS alert was explained, so there is no report "
          "to render.")

### 14.6 Evaluating the explanations

**Purpose** — Quantify how far the explanations themselves can be trusted.

**Input** — The completeness, agreement and stability measurements above.

**Method** — Five families of number, all computed, none asserted.

**Output** — `explanation_metrics.json`.

**Why** — "Explainable" is a claim, and a claim needs evidence like any other.

| What | How it is measured |
|---|---|
| **Completeness** | share of flows within `COMPLETENESS_TOL`, plus the mean and median relative error, per model and target |
| **Stability** | Gaussian noise at `STABILITY_NOISE_STD` (clipped back into $[0,1]$), `STABILITY_N_REPEATS` times; Spearman and top-$K$ overlap against the original ranking |
| **IG vs perturbation** | mean and median Spearman, top-$K$ overlap, and the count of flows that disagree outright |
| **Deletion check** | mean risk drop, top-$K$ against $K$ random features |
| **Fusion consistency** | $\max_x \big| R(x) - \sum_m C_m(x) \big|$ |

**Latency** is reported as **generation time**: IG milliseconds per flow per
model, perturbation per flow, the fusion explanation, and report generation per
alert. It is how long the *code* takes. Nothing here was measured on an analyst,
so no triage time is claimed.

Every setting that could have been tuned -- the baseline, $\tau_s$ -- was chosen
on the **validation** part of the Step 9 split. No test label took part in any
Step 14 setting.

In [ ]:
metrics = step14.get("metrics") or {}
if metrics:
    rows = []
    for model, by_target in (metrics.get("completeness") or {}).items():
        for target, values in by_target.items():
            rows.append({"model": model, "target": target,
                         "within tolerance": values["share_within_tolerance"],
                         "mean relative error": values["mean_relative_error"],
                         "median relative error": values["median_relative_error"],
                         "flows": values["flows"]})
    if rows:
        explainability.show_table(pd.DataFrame(rows), all_columns=True,
            note="Completeness: sum IG against F(x) - F(x'), per flow.")

    stability = metrics.get("stability") or {}
    if stability:
        explainability.show_table(pd.DataFrame([
            {"model": model, **{k: v for k, v in values.items()}}
            for model, values in stability.items()]), all_columns=True,
            note="Stability: the same flows re-attributed under small noise.")

    latency = {k: v for k, v in (metrics.get("latency") or {}).items()
               if isinstance(v, dict)}
    if latency:
        explainability.show_table(pd.DataFrame([
            {"model": model, "IG ms/flow": values["ig_ms_per_flow"],
             "perturbation ms/flow": values["perturbation_ms_per_flow"],
             "baseline risk F(x')": values["baseline_risk"]}
            for model, values in latency.items()]), all_columns=True,
            note="Generation time, not analyst triage time. `baseline risk` is "
                 "F(x') -- it should be low, or the reference flow is not a "
                 "typical normal one.")
        print("fusion explanation: %.3f ms per flow | report generation: "
              "%.3f ms per alert"
              % (metrics["latency"]["fusion_explanation_ms_per_flow"],
                 metrics["latency"]["report_generation_ms_per_alert"]))
    print("temporal attribution:", metrics.get("temporal_attribution"))

### 14.7 Saved outputs and the verification

**Purpose** — Put the evidence on disk and prove the files are right.

**Input** — Everything computed above.

**Method** — Write, then re-read from disk and check.

**Output** — Ten files and seven charts in `step14\`.

**Why** — A table that is correct in memory and wrong on disk is wrong. The
check below reads the saved files back: row counts, $R = \sum_m C_m$, the
presence of every report row, the absence of every forbidden one, and no NaN
anywhere it matters.

In [ ]:
report = step14.get("report")
if report is not None and len(report):
    explainability.show_table(report, all_columns=True,
        note="Read back from disk after writing. A `[FAIL]` here means the "
             "saved file is wrong, whatever the in-memory numbers said.")

thresholds = step14.get("thresholds") or {}
if thresholds:
    explainability.show_table(explainability.kv_table([
        ("tau (Step 13)", "%.6f" % thresholds["tau"]),
        ("tau source", thresholds.get("tau_source")),
        ("tau_s (Step 14, validation)",
         "n/a" if thresholds.get("tau_s") is None
         else "%.6f" % thresholds["tau_s"]),
        ("tau_s validation recall",
         "%.4f" % (thresholds.get("tau_s_choice") or {}).get(
             "validation_recall", float("nan"))),
        ("tau_s validation FPR",
         "%.4f" % (thresholds.get("tau_s_choice") or {}).get(
             "validation_fpr", float("nan"))),
    ]), note="tau is Step 13's and is unchanged. tau_s is chosen on the "
             "validation split only and never touches tau.")

picture = config.STEP14_OUTPUT_DIR / config.STEP14_FILES["chart_bands"]
if picture.exists() and explainability.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(picture)))
elif not picture.exists():
    print("missing:", picture)

folder = config.STEP14_OUTPUT_DIR
if folder.exists():
    print()
    print("Files in", folder)
    for name in sorted(config.STEP14_FILES.values()):
        path = folder / name
        print("  %-36s %s" % (name, "%8.1f KB" % (path.stat().st_size / 1024)
                              if path.exists() else "(not written)"))

### 14.8 Summary

**Purpose** — State what the explanations support, and nothing more.

**Input** — The tables computed above.

**Method** — The text is composed by `explainability.step14_summary()` from
those tables: every figure is read off a computed table, and the verdict on the
deletion check is decided by the numbers rather than written in advance.

**Output** — Rendered here and stored in `explanation_metrics.json`, so the
resumable path shows the same text.

**Why** — A summary written by hand drifts from the tables beside it the moment
anything is re-run.

In [ ]:
text = (step14.get("metrics") or {}).get("summary") or step14.get("summary")
if text:
    if explainability.dl.in_notebook():
        from IPython.display import Markdown, display
        display(Markdown(text))
    else:
        print(text)
else:
    print("No summary stored. Re-run with OVERWRITE_STEP14 = True.")

### What Step 14 can and cannot say

1. **An attribution explains the model, not the world.** It says which input the
   network leaned on. It is not evidence that the feature caused the attack, and
   a model that learned a shortcut is explained faithfully as using that
   shortcut. The right reading of a surprising attribution is to go and check the
   model, not to believe a new fact about attacks.
2. **The attack stage is a lookup of the predicted label.** `ATTACK_STAGE_MAP`
   names a MITRE tactic for a class; an unmapped label reports `Unmapped`.
   Nothing here observes a progression between stages, and no report claims one.
3. **Source context is descriptive, and drawn from a random subset.** It lists
   other test-set alerts from the same Src IP. The test set is a random 20% of
   the flows, so flows in between may simply not be there. No model sees it, and
   it never changes a decision.
4. **The rare classes have few explained alerts.** A per-class attribution for a
   class with a handful of alerts is a handful of observations, not a profile.
5. **Completeness is a necessary check, not a sufficient one.** An attribution
   can add up perfectly and still be a poor guide to the model's behaviour --
   which is exactly why the perturbation comparison and the deletion check are
   run beside it, and why the flows where they disagree are counted rather than
   dropped.
6. **The explanation inherits every limitation of what it explains.** If the
   duplicate-heavy test set flatters the models (Step 12), it flatters their
   explanations too. Step 14 adds no evidence about the data; it only makes the
   models' reasoning visible.

---

## Summary

The pipeline converts raw MSCAD packet captures into labelled flows, reduces 66
CICFlowMeter features to 30 by feature selection and then to 15 by an
autoencoder, and trains a hybrid CNN-LSTM classifier with class weighting applied
to the training loss alone.

**Where the results are written.** Every step writes to its own folder under
`C:\MSCAD2022\outputs\`:

| Artefact | Location |
|---|---|
| Selected features and their ranking | `outputs\step6\selected_features.json`, `combined_ranking.csv` |
| Trained autoencoder and encoder | `outputs\step7\autoencoder.keras`, `encoder.keras` |
| Class weights | `outputs\step9\class_weights.json` |
| Trained classifier and model card | `outputs\step10\cnn_lstm.keras`, `model_card.json` |
| Test metrics and per-class report | `outputs\step11\metrics.json`, `per_class_report.csv` |
| Per-flow predictions | `outputs\step11\predictions.csv` |
| Baseline comparison and charts | `outputs\step12\comparison_table.csv`, `summary.md` |

**How to read the results.** Lead with macro-F1 and G-mean rather than accuracy,
and quote the duplicate-free columns as the conservative estimate. The numeric
results are produced by the cells above and are not reproduced in this text.

**Methodological guarantees enforced by the code.** The scaler is fitted on the
training rows only; feature selection, the autoencoder and the class weights are
derived from training data alone; the test set is never resampled, reweighted or
consulted for any decision before Step 11; and identifier columns are retained
for traceability but excluded from every model input.

**Running this notebook.** The code lives in
`C:\Users\banot\MSCAD22-Multi-Stage-Cyber-Attack-Detection\Notebook`; the data
and all outputs live under `C:\MSCAD2022`. Each step reloads existing outputs
instead of recomputing them unless the corresponding `OVERWRITE_STEPn` flag in
`config.py` is set to `True`.
